## Parcel Update - Spring 2019
* Mason Bindl, mbindl@trpa.org

### Get Parcel Data

* El Dorado County Data:
    * https://gem.edcgov.us/ugotnetextracts/
    * http://gem.edcgov.us/arcgis/rest/services/extracts/geoservices8SQL/MapServer/1

* Placer County Data:
     * email: DDeCelle@placer.ca.gov

* Washoe County Data:
     * http://explore-washoe.opendata.arcgis.com/

* Carson City County Data:
     * http://data-carsoncity.opendata.arcgis.com/datasets/parcels
     * http://www.ccapps.org/profoundui/start?pgm=aspgm/asr840cl
     * Search records FROM APN: 007-011-01 TO APN: 007-031-17 

* Douglas County Data:
     * email: mrichardson@douglasnv.us phone: 775-782-9894

### Import Modules, Set Local Variables, and Define Functions

In [3]:
import arcpy, sys, datetime, os, traceback
# Set Data Source GDB Names
county_gdb = "Original.gdb"
TRPA_staging_gdb = "Staging.gdb"
w_folder = '2019_04'  # Change based on new file folder name
w_original = '\\Original\\'
w_modified = '\\Modified\\'
n_parcels = 'Parcels'
sep = '-'
County_List = ['Carson', 'Douglas', 'El_Dorado', 'Placer', 'Washoe']
County_ABR_List = ['CC', 'DG', 'EL', 'PL', 'WA']
state_loc = ['CA', 'NV']
python_version = "PYTHON3"

# set base feature classes
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
ParcelPoint = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Point"

# if arcpy.Exists(ParcelPoint):
#     arcpy.Delete_management(ParcelPoint)
# arcpy.FeatureToPoint_management(ParcelLayer, ParcelPoint, "INSIDE")

# SDE feature classes from C: drive
# sde_FireDistrict = "C:\\GIS\\PROJECT\\SDE\\SDE_Archive_2018.gdb\\Jurisdictions\\FireDistricts"

# File paths
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"
#sde feature classes
sde_FireDistrict = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_NRCSSoils1974 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_HydroArea = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed = sdeBase + "\\sde.SDE.Water\\sde.SDE.Priority"
sde_RegionalLandUse = sdeBase + "\\sde.SDE.Planning\\sde.SDE.RegionalLandUse"
sde_LocalPlan = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_Zoning =  sdeBase + "\\sde.SDE.Planning\\sde.SDE.Zoning_LocalPlan"
sde_SpecialDistrict = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_TownCenter = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987 = sdeBase + "\\sde.SDE.Planning\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_UrbanArea = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_CSLT = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"

# in memory files
wk_memory = "in_memory" + "\\"
ParcelPoint_FireDistrict = wk_memory + "\\ParcelPoint_FireDistrict"
ParcelPoint_Soils74 = wk_memory + "\\ParcelPoint_Soils74"
ParcelPoint_Soils03 = wk_memory + "\\ParcelPoint_Soils03"
ParcelPoint_HydroArea = wk_memory + "\\ParcelPoint_HydroArea"
ParcelPoint_Watershed = wk_memory + "\\ParcelPoint_Watershed"
ParcelPoint_RegionalLandUse = wk_memory + "\\ParcelPoint_RegionalLandUse"
ParcelPoint_LocalPlan = wk_memory + "\\ParcelPoint_LocalPlan"
ParcelPoint_TownCenter = wk_memory + "\\ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer = wk_memory + "\\ParcelPoint_TownCenterBuffer"
ParcelPoint_Zoning = wk_memory + "\\ParcelPoint_Zoning"
ParcelPoint_SpecialDistrict = wk_memory + "\\ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987 = wk_memory + "\\ParcelPoint_Index1987"
ParcelPoint_PstlTown = wk_memory + "\\ParcelPoint_PstlTown"
ParcelPoint_PstlZip = wk_memory + "\\ParcelPoint_PstlZip"
ParcelPoint_CSLT = wk_memory + "\\ParcelPoint_CSLT"

## Translate County Data

### Select County to be proccessed to set field list

In [ ]:
while True:
    try:
        UserCountyResponse = input("Enter a County Name: ")
        if UserCountyResponse.lower() == "carson":
            UserCountyResponse = (County_List[0])
            County_ABR = (County_ABR_List[0])
            fld_list = ['APN', 'APN', 'Legal_Owner', 'Mail_Addr', 'Mail2_Addr', 'MCity', 'MZip', 'Land_Value',
                        'Improv_Val', 'Loc1', 'Unit', 'Dir', 'Phy_Addr', 'UPDATED', 'LU']
            break
        elif UserCountyResponse.lower() == "douglas":
            UserCountyResponse = (County_List[1])
            County_ABR = (County_ABR_List[1])
            fld_list = ['TAG', 'PPNO', 'PANAME', 'PMADD2', 'PMADD1', 'PMCTST', 'PZIP', 'YLANDV',
                        'YIMPRV', 'PLOC_', 'PLOCU_', 'PLOCDR', 'PLOCNM', 'PUPDDT', 'YLDUSE', 'PLOCTP', 'PAPPYR', 'PTOWN',
                        'POLDP_', 'POLDDT']
            break
        elif UserCountyResponse.lower() == "el_dorado" or UserCountyResponse.lower() == "el dorado":
            UserCountyResponse = (County_List[2])
            County_ABR = (County_ABR_List[2])
            fld_list = ['PRCL_ID', 'PRCL_ID', 'OWNER_NAME', 'OWNER_ADDR', 'OWNER_CITY', 'OWNER_STAT', 'OWNER_ZIP',
                        'LANDVAL', 'STRUCVAL', 'SITUSNUMBR', 'UNIT_NBR', 'SITUSSTRNM', 'SITUSSTRTY', 'POLY_CREAT',
                        'USECDPRI', 'CITY', 'LANDVAL', 'STRUCVAL']
            break
        elif UserCountyResponse.lower() == "placer":
            UserCountyResponse = (County_List[3])
            County_ABR = (County_ABR_List[3])
            fld_list = ['APN', 'GISAPN', 'OWNER1', 'ADR1', 'ADR2', 'CITY', 'STATE', 'ZIP', 'LANDVALUE', 'STRUCTURE',
                        'STREETNUM', 'SP_APT', 'STREETDIR', 'STREETNAME', 'STREETTYPE', 'USE_CD', 'COMMUNITY', 'TAXYEAR', 'TRANSACTIO']
            break
        elif UserCountyResponse.lower() == "washoe":
            UserCountyResponse = (County_List[4])
            County_ABR = (County_ABR_List[4])
            fld_list = ['PIN', 'APN', 'FIRSTNAME', 'LASTNAME', 'MAILING1', 'MAILING2', 'MAILCITY', 'MAILSTATE',
                        'MAILZIP', 'LANDASS', 'BUILDASS',
                        'STREETNUM', 'STREETDIR', 'STREET', 'LAND_USE', 'TAXYEAR', 'CITY', 'SITUSZIP']
            break
        else:
            print ("County entered is not valid. Try a county listed below")
            for x in County_List:
                print (x)
            print('\n')
            continue
    except Exception as e:
        print (e)
        sys.exit()
print ("{} County will be processed".format(UserCountyResponse))

#### Set Workspace Enviroment

In [28]:
base_working_folder = os.path.join(r'F:\GIS\ParcelUpdate', w_folder, UserCountyResponse)
arcpy.env.workspace = base_working_folder + w_original + county_gdb
print (arcpy.env.workspace)

F:\GIS\ParcelUpdate\2019_04\Washoe\Original\Original.gdb


#### Check for ArcInfo license

In [29]:
try:
    license_info = arcpy.ProductInfo()
    if not license_info == "NotInitialized":
        print ("Product License: {}".format(arcpy.ProductInfo()))
    else:
        raise Exception
except Exception:
    print ("ArcGIS Product License Error: {}".format(arcpy.ProductInfo()))
    sys.exit()

Product License: ArcInfo


#### Set Default Extent

In [30]:
arcpy.env.extent = sde_TRPAboundary
print ("The feature extent is being used in env {}".format(str(arcpy.env.extent)))
arcpy.env.spatialGrid1 = s_grid_1 = 5400
arcpy.env.spatialGrid2 = s_grid_2 = 16200
arcpy.env.spatialGrid3 = s_grid_3 = 48600

The feature extent is being used in env 737668.911313948 4288193.03147232 770952.921734459 4357302.88519489 NaN NaN NaN NaN


#### Set Default Coordinate System

In [31]:
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 UTM Zone 10N")
sr = arcpy.env.outputCoordinateSystem
print ("Spatial Reference: {}".format(sr.name))

Spatial Reference: NAD_1983_UTM_Zone_10N


#### Remove M and Z values

In [32]:
arcpy.env.outputZFlag = "Disabled"
arcpy.env.outputMFlag = "Disabled"
arcpy.env.overwriteOutput = True

#### Set Local Variables

In [33]:
# Set local variables
out_gdb_name = County_ABR + "_" + TRPA_staging_gdb 
modified_folder = base_working_folder + w_modified
original_folder = base_working_folder + w_original
modified_fgd = modified_folder + out_gdb_name
original_fgd = original_folder + out_gdb_name
# County local variables
parcel_in_data = UserCountyResponse + '_' + n_parcels
fc_outname = parcel_in_data
fdata_name = n_parcels
fdataset = os.path.join(modified_fgd, n_parcels)
parcel_out_data = fdataset + '\\' + fc_outname
print ("Features saved here: {}".format(parcel_out_data))

Features saved here: F:\GIS\ParcelUpdate\2019_04\Washoe\Modified\WA_Staging.gdb\Parcels\Washoe_Parcels


#### Track Progress

In [34]:
# Start timer for process
start_time = datetime.datetime.today()
print ("Script started: {}".format(start_time))
# Create  and open log file.
complete_txt_path = os.path.join(base_working_folder, fc_outname + ".txt")
print (complete_txt_path)
log = open(complete_txt_path, "w")
# Write results to txt file
log.write("Log: " + str(start_time) + "\n")
log.write("\n")
log.write("Begin process:\n")
log.write("Process started at: " + str(start_time) + "\n")
log.write("\n")

Script started: 2019-04-22 12:11:09.292839
F:\GIS\ParcelUpdate\2019_04\Washoe\Washoe_Parcels.txt


1

#### Describe original feature class and its spatial reference

In [35]:
desc = arcpy.Describe(parcel_in_data)
spatialref = desc.spatialReference
print ("Original Spatial Reference: {}".format(spatialref.name))
log.write("Original Spatial Reference: {}".format(spatialref.name) + "\n")

Original Spatial Reference: NAD_1983_StatePlane_Nevada_West_FIPS_2703_Feet


75

#### Create File GDB and Feature Class for Chosen County

In [36]:
while True:
    try:
        CreateGDBResponse = input("Create new GDB and Feature Class? (Yes/No)")
        if CreateGDBResponse.lower() == "yes":
            # Check if FileGDB exists. If not Execute CreateFileGDB
            if not arcpy.Exists(modified_fgd):
                arcpy.CreateFileGDB_management(modified_folder, out_gdb_name)
                print ("File Geodatabase {} created!".format(out_gdb_name))
                log.write("File Geodatabase {} created!".format(out_gdb_name) + "\n")
            # Execute CreateFileGDB will not be run
            else:
                print ("File Geodatabase {} already exists!".format(out_gdb_name))
                log.write("File Geodatabase {} already exists!".format(out_gdb_name) + "\n")
            # Check if Feature Dataset exists. Execute Copy
            if not arcpy.Exists(fdataset):
                arcpy.CreateFeatureDataset_management(modified_fgd, fdata_name, sr)
                print ("Feature Dataset Created!")
            # Execute Delete Management then execute CopyFeatures Management
            else:
                print ("Feature Dataset {} already exists!".format(fdata_name))
                arcpy.Delete_management(fdataset)
                print ("Deleted {}!".format(fdata_name))
                log.write("Deleted {}".format(fdata_name) + "\n")
                arcpy.CreateFeatureDataset_management(modified_fgd, fdata_name, sr)
                print ("Created Dataset {}".format(fdata_name))
            fcs = arcpy.ListFeatureClasses()
            for fc in fcs:
                # Copy features from the workspace to FGD
                arcpy.CopyFeatures_management(fc, parcel_out_data)
                print ("Added {}".format(fc))
                log.write("Added {}".format(fc) + "\n")
                # Execute GP tool to get count of records in FeatureClass
                result = arcpy.GetCount_management(parcel_out_data)
                oldresult = arcpy.GetCount_management(parcel_in_data)
                oldcount = int(oldresult.getOutput(0))
                count = int(result.getOutput(0))
                print ("Number of records copied {0} out of the {1}".format(count, oldcount))
                log.write("Number of records copied {0} out of the {1}".format(count, oldcount) + "\n")
                if count > 0:
                    # Create list of field names in FC that is being processed
                    Fc_fields = arcpy.ListFields(parcel_out_data)
                    for field in Fc_fields:
                        print ("{0} type: {1} length: {2}".format(field.name, field.type, field.length))
                        log.write("Name: {}".format(field.name) + ", ")
                        log.write("Type: {}".format(field.type) + ", ")
                        log.write("Length: {}".format(field.length) + "\n")
                    # Add all fields
                    fields = [
                        # Fields for APN & PPNO
                        ("APN_NEW", "TEXT", "", "", "16", "APN", "NULLABLE", "NON_REQUIRED", ""),
                        ("PPNO_NEW", "DOUBLE", "", "0", "", "PPNO", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for local address table
                        ("HSE_NUMBR_NEW", "SHORT", "", "", "5", "Lot #", "NULLABLE", "NON_REQUIRED", ""),
                        ("UNIT_NUMBR_NEW", "Text", "", "", "12", "Unit #", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_DIR_NEW", "Text", "", "", "2", "St Direction", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_NAME_NEW", "Text", "", "", "100", "St Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("STR_SUFFIX_NEW", "Text", "", "", "6", "St Suffix", "NULLABLE", "NON_REQUIRED", ""),
                        ("APO_ADDRESS_NEW", "Text", "", "", "100", "Full Address", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_TOWN_NEW", "TEXT", "", "", "25", "Town", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_STATE_NEW", "TEXT", "", "", "2", "State", "NULLABLE", "NON_REQUIRED", ""),
                        ("PSTL_ZIP5_NEW", "TEXT", "", "", "5", "ZIP5", "NULLABLE", "NON_REQUIRED", ""),
                        # fields for mailing address
                        ("OWN_FIRST_NEW", "TEXT", "", "", "50", "Owner First Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWN_LAST_NEW", "TEXT", "", "", "100", "Owner Last Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWN_FULL_NEW", "TEXT", "", "", "100", "Owner Full Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ADD1_NEW", "TEXT", "", "", "100", "Mail Address 1", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ADD2_NEW", "TEXT", "", "", "100", "Mail Address 2", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_CITY_NEW", "TEXT", "", "", "50", "Mail City", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_STATE_NEW", "TEXT", "", "", "2", "Mail State", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAIL_ZIP5_NEW", "TEXT", "", "", "5", "Mail ZIP5", "NULLABLE", "NON_REQUIRED", ""),
                        ("JURISDICTION_NEW", "TEXT", "", "", "4", "Jurisdiction", "NULLABLE", "NON_REQUIRED", ""),
                        ("COUNTY", "TEXT", "", "", "2", "County", "NULLABLE", "NON_REQUIRED", ""),
                        ("OWNERSHIP_TYPE", "TEXT", "", "", "12", "Ownership  Type", "NULLABLE", "NON_REQUIRED", ""),
                        ("UNITS_NEW", "SHORT", "", "", "1", "Units", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for land use, soil, watershed, etc...
                        ("COUNTY_LANDUSE_CODE", "TEXT", "", "", "4", "County Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("COUNTY_LANDUSE_DESCRIPTION", "TEXT", "", "", "150", "County Land Use Description", "NULLABLE", "NON_REQUIRED", ""),
                        ("TRPA_LANDUSE_DESCRIPTION", "TEXT", "", "", "50", "TRPA Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("REGIONAL_LANDUSE", "TEXT", "", "", "50", "TRPA Regional Land Use", "NULLABLE", "NON_REQUIRED", ""),
                        ("SOIL_1974", "TEXT", "", "", "5", "Soils 1974", "NULLABLE", "NON_REQUIRED", ""),
                        ("SOIL_2003", "TEXT", "", "", "5", "Soils 2003", "NULLABLE", "NON_REQUIRED", ""),
                        ("ALLOWABLE_COVERAGE_BAILEY_SQFT", "DOUBLE", "12", "2", "", "Allowable Coverage (Bailey)", "NULLABLE", "NON_REQUIRED", ""),
                        ("IMPERVIOUS_SURFACE_SQFT", "DOUBLE", "12", "2", "", "Impervious Surface (LiDAR, sq.ft.)", "NULLABLE", "NON_REQUIRED", ""),
                        ("HRA_NAME", "TEXT", "", "", "30", "HRA Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("WATERSHED_NUMBER", "SHORT", "", "", "4", "Watershed #", "NULLABLE", "NON_REQUIRED", ""),
                        ("WATERSHED_NAME", "TEXT", "", "", "30", "Watershed Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("PRIORITY_WATERSHED", "TEXT", "", "", "2", "Priority Watershed", "NULLABLE", "NON_REQUIRED", ""),
                        #("BMP_STATUS", "SHORT", "", "", "1", "BMP Status", "NULLABLE", "NON_REQUIRED", ""),
                        ("FIREPD", "TEXT", "", "", "25", "Fire Protection District", "NULLABLE", "NON_REQUIRED", ""),
                        ("WITHIN_TRPA_BNDY", "SHORT", "", "", "1", "Within TRPA Boundary", "NULLABLE", "NON_REQUIRED", ""),
                        ("LITTORAL", "SHORT", "", "", "1", "Littoral", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Parcel Value table
                        ("AS_LANDVALUE_NEW", "LONG", "9", "", "", "Assessed Land", "NULLABLE", "NON_REQUIRED", ""),
                        ("AS_IMPROVALUE_NEW", "LONG", "9", "", "", "Assessed Improvement", "NULLABLE", "NON_REQUIRED", ""),
                        ("AS_SUM_NEW", "LONG", "9", "", "", "Assessed Sum", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_LANDVALUE_NEW", "LONG", "9", "", "", "Tax Land", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_IMPROVALUE_NEW", "LONG", "9", "", "", "Tax Improvement", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_SUM_NEW", "LONG", "9", "", "", "Tax Sum", "NULLABLE", "NON_REQUIRED", ""),
                        ("TAX_YEAR_NEW", "TEXT", "", "", "4", "Tax Year", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Planning purposes
                        ("PAS_ID", "TEXT", "", "", "8", "PAS ID", "NULLABLE", "NON_REQUIRED", ""),
                        ("PAS_NAME", "TEXT", "", "", "40", "PAS Name", "NULLABLE", "NON_REQUIRED", ""),
                        ("INDEX_1987", "TEXT", "", "", "10", "1987 Parcel Map Index", "NULLABLE", "NON_REQUIRED", ""),
                        ("INDEX_1987_HYPERLINK", "TEXT", "", "", "255", "1987 Parcel Map Index Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("LOCAL_PLAN_HYPERLINK", "TEXT", "", "", "255", "Local Plan Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("DESIGN_GUIDELINES_HYPERLINK", "TEXT", "", "", "255", "Design Guideline Hyperlink", "NULLABLE", "NON_REQUIRED", ""),
                        ("ZONING_NEW", "TEXT", "", "", "50", "Zoning", "NULLABLE", "NON_REQUIRED", ""),
                        ("ZONING_DESCRIPTION", "TEXT", "", "", "50", "Zoning Description", "NULLABLE", "NON_REQUIRED", ""),
                        ("SINGLE_FAMILY_DENSITY", "TEXT", "", "", "50", "Single Family Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("MULTI_FAMILY_DENSITY", "TEXT", "", "", "50", "Multi-Family Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("TOURIST_ACCOMMODATION_DENSITY", "TEXT", "", "", "50", "Tourist Accommodation Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("BED_BREAKFAST_DENSITY", "TEXT", "", "", "50", "Bed & Breakfast Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("TIME_SHARE_DENSITY", "TEXT", "", "", "50", "Time Share Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("COMMERCIAL_FLOOR_AREA_ALLOWED", "TEXT", "", "", "50", "Commercial Floor Area Allowed", "NULLABLE", "NON_REQUIRED", ""),
                        ("SECONDARY_DWELLING_UNIT_ALLOWED", "TEXT", "", "", "150", "Secondary Dwelling Unit Allowed", "NULLABLE", "NON_REQUIRED", ""),
                        ("OVERLAY", "TEXT", "", "", "50", "Overlay", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAX_CP_RES_DENSITY", "TEXT", "", "", "50", "Max CP RES Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("MAX_CP_TAU_DENSITY", "TEXT", "", "", "50", "Max CP TAU Density", "NULLABLE", "NON_REQUIRED", ""),
                        ("SPECIAL_PLAN_AREA_OVERLAY", "TEXT", "", "", "50", "Special Pland Area Overylay", "NULLABLE", "NON_REQUIRED", ""),
                        ("TOWN_CENTER", "TEXT", "", "", "50", "Town Center", "NULLABLE", "NON_REQUIRED", ""),
                        ("SPECIAL_AREA", "TEXT", "", "", "50", "Special Area", "NULLABLE", "NON_REQUIRED", ""),
                        ("SPECIAL_PLANNING_DISTRICT", "TEXT", "", "", "50", "Special Planning District", "NULLABLE", "NON_REQUIRED", ""),
                        ("SPECIAL_PLANNING_DISTRICT_TYPE", "TEXT", "", "", "50", "Special Planning District Type", "NULLABLE", "NON_REQUIRED", ""),
                        ("LOCATION_TO_TOWNCENTER", "TEXT", "", "", "50", "Location Relative to Town Center", "NULLABLE", "NON_REQUIRED", ""),
                        ("RETIRED", "TEXT", "", "", "50", "Retired", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Parcel Size
                        ("PARCEL_ACRES_NEW", "DOUBLE", "6", "2", "", "Acres", "NULLABLE", "NON_REQUIRED", ""),
                        ("PARCEL_SQFT_NEW", "DOUBLE", "12", "2", "", "Square Feet", "NULLABLE", "NON_REQUIRED", ""),
                        # Fields for Parcel History
                        ##("PRCL_DT_UPDATE", "Date", "", "", "", "Parcel Date Change", "NULLABLE", "NON_REQUIRED", ""),
                        ##("PREV_APN_STRD", "TEXT", "", "", "16", "Previous APN", "NULLABLE", "NON_REQUIRED", ""),
                        ##("PREV_PPNO_STRD", "DOUBLE", "11", "0", "", "Previous PPNO", "NULLABLE", "NON_REQUIRED", ""),
                        ##("APN_DT_CHANGE", "Date", "", "", "", "APN Date Change", "NULLABLE", "NON_REQUIRED", ""),
                        # Field for finding duplicates
                        ("DUPLICATE", "SHORT", "", "", "1", "", "NULLABLE", "NON_REQUIRED", "")
                    ]
                    # Create the fields using the above parameters
                    for field in fields:
                        arcpy.AddField_management(*(parcel_out_data,) + field)
                        print("{} field created".format(field))
                    break
                else:
                    print("{} No features in feature class. Exit Script".format(count))
                    exit()
            break
        elif CreateGDBResponse.lower() == "no":
            # Check if FileGDB exists.
            if arcpy.Exists(modified_fgd):
                print ("It's a go!")
                break
            else:
                print ("Staging Geodatabase doesn't exist. Needs to be created. Type Yes")
                continue
        else:
            print ("Staging Geodatabase doesn't exist. Needs to be created. Type Yes")
            continue
    except Exception as e:
        print (e)
        exit()

# print "Begin list of all fields that exists"
Strd_fields = arcpy.ListFields(parcel_out_data)
for field in Strd_fields:
    # print ("{0} type: {1} length: {2}".format(field.name, field.type, field.length))
    log.write("Name: {}".format(field.name) + ", ")
    log.write("Type: {}".format(field.type) + ", ")
    log.write("Length: {}".format(field.length) + "\n")

# Get the list of field names in file and list of predefined field names
# that will be calculated. Missing fields will not crash script, only close.
field_names = [field.name for field in arcpy.ListFields(parcel_out_data)]
match_fields = [i for i in fld_list if i in field_names]
unmatch_fields = [i for i in fld_list if i not in field_names]


Create new GDB and Feature Class? (Yes/No)Yes
File Geodatabase WA_Staging.gdb already exists!
Feature Dataset Parcels already exists!
Deleted Parcels!
Created Dataset Parcels
Added Washoe_Parcels
Number of records copied 9422 out of the 9422
OBJECTID_1 type: OID length: 4
Shape type: Geometry length: 0
OBJECTID type: Integer length: 4
PERIMETER type: String length: 80
APN type: Integer length: 4
REGION type: String length: 80
PIN type: String length: 80
RECMAP type: String length: 80
BOOK type: String length: 80
PAGE type: String length: 80
BLOCK type: String length: 80
PARCEL type: String length: 80
SUBNAME type: String length: 80
TOWNSHIP type: String length: 80
RANGE type: String length: 80
SECTION_ type: String length: 80
FLOOR type: Integer length: 4
MAPLINK type: String length: 80
FLR type: Integer length: 4
STREETNUM type: String length: 80
STREETDIR type: String length: 80
STREET type: String length: 80
CITY type: String length: 80
SITUSZIP type: String length: 80
FIRSTNAME typ

('SINGLE_FAMILY_DENSITY', 'TEXT', '', '', '50', 'Single Family Density', 'NULLABLE', 'NON_REQUIRED', '') field created
('MULTI_FAMILY_DENSITY', 'TEXT', '', '', '50', 'Multi-Family Density', 'NULLABLE', 'NON_REQUIRED', '') field created
('TOURIST_ACCOMMODATION_DENSITY', 'TEXT', '', '', '50', 'Tourist Accommodation Density', 'NULLABLE', 'NON_REQUIRED', '') field created
('BED_BREAKFAST_DENSITY', 'TEXT', '', '', '50', 'Bed & Breakfast Density', 'NULLABLE', 'NON_REQUIRED', '') field created
('TIME_SHARE_DENSITY', 'TEXT', '', '', '50', 'Time Share Density', 'NULLABLE', 'NON_REQUIRED', '') field created
('COMMERCIAL_FLOOR_AREA_ALLOWED', 'TEXT', '', '', '50', 'Commercial Floor Area Allowed', 'NULLABLE', 'NON_REQUIRED', '') field created
('SECONDARY_DWELLING_UNIT_ALLOWED', 'TEXT', '', '', '150', 'Secondary Dwelling Unit Allowed', 'NULLABLE', 'NON_REQUIRED', '') field created
('OVERLAY', 'TEXT', '', '', '50', 'Overlay', 'NULLABLE', 'NON_REQUIRED', '') field created
('MAX_CP_RES_DENSITY', 'TEXT'

#### Get Year

In [37]:
# Get current year
current_year = datetime.datetime.today().year

#### List of variation of all federal, state and local agencies found in assessor datasets

In [38]:
fedOwnList = ("USA FOREST SERVICE", "USDA FOREST SERVICE", "USDA - FOREST SERVICE", "UNITED STATES POSTAL", 
              "UNITED STATES OF AMERICA", "UNITED STATES FOREST SERVICE", "U S POSTAL SERVICE", "U S COAST GUARD",
              "U S A FOREST SERVICE``", "U S A FOREST SERVICE", "LAKE VALLEY RANGER STA", "DEPT OF VETRANS AFFAIRS%", 
              "DEPT OF VETERANS AFFAIRS%", "DEPT OF VETERANS AFFAIRS %", "BUREAU OF LAND MANAGEMENT", "U S FOREST SERVICE")

stateOwnList = ("TAHOE CONSERVANCY", "STATE OF NEVADA FOREST SERVICE", "STATE OF NEVADA", "STATE OF CALIFORNIA THE", 
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "STATE OF CA", "REGENTS OF UNIV OF CALIF", 
                "NEVADA, STATE OF", "NEVADA STATE OF", "CALIFORNIA TAHOE CONSERVANCY ET AL", "CALIFORNIA TAHOE CONSERVANCY"
                "CALIFORNIA STATE OF THE", "CALIFORNIA STATE OF ET AL", "CALIFORNIA STATE OF")

localOwnList = ("ZEPHYR COVE GENERAL IMP DIST", "WASHOE COUNTY SCHOOL DISTRICT BOARD", "WASHOE COUNTY", "WASHOE TRIBE OF NV & CA", 
                "TALMONT RESORT IMPROVEMENT DISTRICT", "TALMONT RESORT IMPR DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMP DIST", "TAHOE PARADISE RESORT IMP DIST", "TAHOE PARADISE RES IMP DST",
                "TAHOE FOREST HOSPITAL DISTRICT", "TAHOE TRUCKEE UNIFIED SCHOOL DISTRICT", "TAHOE TRUCKEE UNIFIED SCH DIST", 
                "TAHOE DOUGLAS FIRE PROTECT DIST", "TAHOE DOUGLAS SEWER DIST", "TAHOE DOUGLAS DISTRICT", 
                "TAHOE CITY PUBLIC UTILITY DISTRICT", "TAHOE CITY PUBLIC UTILITY DIST", "TAHOE CITY PUBLIC UTILDIST", 
                "TAHOE CITY PUB UTILITY DST", "TAHOE CITY PUB UTILITY DIS", "TAHOE CITY P U D", "TAHOE CITY CEMETERY DIST", 
                "SOUTH TAHOE REDEVELP AGENCY", "SOUTH TAHOE REFUSE CO", "SOUTH TAHOE PUD", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTH TAHOE PUBLIC UTILITYDIST", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTILITY DIS", 
                "SOUTH TAHOE PUBLIC UTILITY", "SOUTH TAHOE PUBLIC UTIL DT", "SOUTH TAHOE PUBLIC UTIL DIST", 
                "SOUTH TAHOE PUBLIC", "SOUTH TAHOE PUB UTIL DIST", "SOUTH LAKE TAHOE CTYOF 1/3", "SOUTH LAKE TAHOE CITY OF", 
                "SO TAHOE PUBLIC UTILITY DIST", "SO TAHOE PUB UTIL DIST", "SIERRA NEVADA COLLEGE", "ROUND HILL GEN IMP DIST",
                "PLACER COUNTY REDEVELOPMENT AGENCY", "PLACER COUNTY OF", "PLACER COUNTY", "NORTH TAHOE PUBLIC UTL DIST",
                "NORTH TAHOE PUBLIC UTILITY DISTRICT", "NORTH TAHOE PUBLIC UTILITY DIST", "NORTH TAHOE PUBLIC UTILITY DIS", 
                "NORTH TAHOE PUBLIC UTILITIES DIST", "NORTH TAHOE PUBLIC UTILIITY DISTRICT", "NORTH TAHOE P U D",
                "NORTH TAHOE FIRE PROTECTION DISTRICT", "NORTH TAHOE FIRE PROTECTION", "NORTH TAHOE FIRE DIST",
                "NORTH LAKE TAHOE FIRE PROTECTION DIST", "N TAHOE FIRE PROTECTION DIST", "MEEKS BAY FIRE PROT DIST", 
                "LAKERIDGE GENERAL IMP DIST", "LAKE VALLEY FIRE PROTECTION", "LAKE VALLEY FIRE PROT DST", "LAKE VALLEY FIRE PROT DIST", 
                "LAKE VALLEY FIRE DISTRICT", "LAKE TAHOE UNIFIED SCHOOL DIST", "LAKE TAHOE SCHOOL", "LAKERIDGE GENERAL IMP DIST", 
                "LAKE TAHOE FIRE PROTECTION DIST", "LAKE TAHOE FIRE PROTECT DIST", "LAKE TAHOE COMM COLLEGE DIST",
                "LAKE TAHOE COMM COL DIST", "KINGSBURY GENERAL IMP DISTRICT", "KINGSBURY GENERAL IMP DIST",
                "INCLINE VILLAGE GENERAL IMPROVEMENT DISTRICT", "INCLINE VILLAGE GENERAL IMPROVEMENT DIST", 
                "DOUGLAS COUNTY SEWER DIST", "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS COUNTY", "DOUGLAS CO SEWER IMP DIST #1", 
                "COUNTY OF EL DORADO", "CITY OF SOUTH LAKE TAHOE", "EL DORADO IRRIGATION DISTRICT", 
                "HAPPY HOMESTEAD CEMETERY DIST") 

ownerListVal = ("STATE OF NEVADA", "STATE OF NEVADA FOREST SERVICE", "NEVADA, STATE OF",
                "NEVADA, DEPT OF TRANSPORTATION",
                "STATE OF CALIFORNIA (EASEMENT)", "STATE OF CALIFORNIA", "CALIFORNIA STATE OF", "STATE OF CALIFORINA",
                "STATE OF CALIFORIA", "STATE OF CA", "CALIFORNIA STATE OF ET AL",
                "UNITED STATES OF AMERICA", "U S A FOREST SERVICE", "U S A  FOREST SERVICE", "USA FOREST SERVICE",
                "U S COAST GUARD", "BUREAU OF LAND MANAGEMENT", "DEPT OF VETRANS AFFAIRS", "DEPT OF VETERANS AFFAIRS",
                "DEPT OF VETERANS AFFAIRS %",
                "U S FOREST SERVICE", "UNITED STATES FOREST SERVICE", "UNITED STATES POSTAL",
                "CARSON CITY", "DOUGLAS COUNTY",
                "CITY OF SOUTH LAKE TAHOE", "COUNTY OF EL DORADO", "EL DORADO COUNTY OF", "PLACER COUNTY",
                "PLACER COUNTY OF", "PLACER COUNTY REDEVELOPMENT AGENCY",
                "DOUGLAS COUNTY SCHOOL DIST", "DOUGLAS CO SEWER IMP DIST #1", "LAKE TAHOE UNIFIED SCHOOL DIST",
                "TALMONT RESORT IMP DIST", "TALMONT RESORT IMP DISTRICT",
                "TALMONT RESORT IMPROVEMENT DISTRICT", "CALIFORNIA TAHOE CONSERVANCY", "CA TAHOE CONSERVANCY",
                "LAKE TAHOE COMM COL DIST",
                "SO TAHOE PUB UTIL DIST", "SO TAHOE PUBLIC UTILITY DIST", "SOUTH TAHOE PUB UTIL DIST",
                "SOUTH TAHOE PUBLIC UTIL DIST", "SOUTH TAHOE PUBLIC UTIL DT",
                "SOUTH TAHOE PUBLIC UTILITY DIS", "SOUTH TAHOE PUBLIC UTILITY DST", "SOUTH TAHOE PUBLIC UTL DST",
                "SOUTHTAHOE PUBLIC UTILITY DIST", "CALIFORNIA TAHOE CONSERVANCY ET AL")


#### List all field names and their index number

In [39]:
field_names = [field.name for field in arcpy.ListFields(parcel_out_data)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

0 OBJECTID_1
1 Shape
2 OBJECTID
3 PERIMETER
4 APN
5 REGION
6 PIN
7 RECMAP
8 BOOK
9 PAGE
10 BLOCK
11 PARCEL
12 SUBNAME
13 TOWNSHIP
14 RANGE
15 SECTION_
16 FLOOR
17 MAPLINK
18 FLR
19 STREETNUM
20 STREETDIR
21 STREET
22 CITY
23 SITUSZIP
24 FIRSTNAME
25 LASTNAME
26 MAILING1
27 MAILING2
28 MAILCITY
29 MAILSTATE
30 MAILZIP
31 LAND_USE
32 ZONING
33 WATER
34 SEWER
35 ACREAGE
36 TAXDIST
37 BEDROOMS
38 BATHS
39 YEARBLT
40 LANDASS
41 BUILDASS
42 TOTALASS
43 LANDAPR
44 BUILDAPR
45 TOTALAPR
46 DEPRECIATI
47 SALEDATE
48 SALEPRICE
49 OCCUPANCY
50 PROPCODE
51 TAXYEAR
52 STORIES
53 TOWNHOUSE
54 SQFEET
55 UNITS
56 AVCONSYEAR
57 BUILDINGTY
58 NBHD
59 SPECPROPCO
60 QC
61 LAND_BASE
62 NBC
63 Fireplaces
64 HeatType
65 SecHeatTyp
66 Agency
67 AG_ABBR
68 Abate
69 Exempt
70 Secured
71 NetTax
72 GROSSTAX
73 TAXVALUE
74 dor_cd
75 FullAddres
76 ZoningWith
77 Shape_Length
78 Shape_Area
79 APN_NEW
80 PPNO_NEW
81 HSE_NUMBR_NEW
82 UNIT_NUMBR_NEW
83 STR_DIR_NEW
84 STR_NAME_NEW
85 STR_SUFFIX_NEW
86 APO_ADDRESS_NEW
87 P

### Translate County Values to TRPA Values and Fields

In [41]:
# process Washoe County fields
if UserCountyResponse == "Washoe":
    print ("Processing the {} dataset".format(County_List[4]))
    field_names = ['PIN', 'APN_NEW', 'APN', 'PPNO_NEW',
                   'STREETNUM','HSE_NUMBR_NEW','UNIT_NUMBR_NEW', 'STREETDIR', 'STR_DIR_NEW',
                   'STREET', 'STR_NAME_NEW', 'STR_SUFFIX_NEW','PSTL_STATE_NEW',
                   'FIRSTNAME', 'OWN_FIRST_NEW', 'LASTNAME', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'MAILING1','MAIL_ADD1_NEW', 'MAILING2','MAIL_ADD2_NEW',
                   'MAILCITY', 'MAIL_CITY_NEW', 'MAILSTATE','MAIL_STATE_NEW', 
                   'MAILZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE',
                   'LAND_USE', 'COUNTY_LANDUSE_CODE', 
                   'LANDASS', 'AS_LANDVALUE_NEW','BUILDASS','AS_IMPROVALUE_NEW','AS_SUM_NEW',
                   'LANDAPR','TAX_LANDVALUE_NEW','BUILDAPR','TAX_IMPROVALUE_NEW','TAX_SUM_NEW','TAX_YEAR_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True):
                row[1] = apn
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not ppno is None:
                row[3] = int(ppno)
            else:
                row[3] = ""
            # set house number
            hsenumbr = row[4]
            if not (hsenumbr is None or hsenumbr == "" or hsenumbr.isspace() == True):
                row[5] = (hsenumbr.strip())
            else:
                row[5] = 0
            # set unit number (doesn't exist in WA data)
            row[6] = ""
            # set street direction
            stdir = row[7]
            if not stdir is None:
                row[8] = (stdir.strip())
            else:
                row[8] = ""
            # set street name    
            stname = row[9]
            if not (stname is None or stname in ('CROSS BOW', 'ENTERPRISE','STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395 S', '')):
                row[10] = (stname.rsplit(" ", 1)[0].strip())
            elif stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395 S', ''):
                row[10] = (stname.strip())
            else:
                row[10] = ""
            # set street suffix
            stname = row[9]
            if not stname in ('CROSS BOW', 'ENTERPRISE', 'STATE ROUTE 28', 'UNSPECIFIED', 'US HIGHWAY 395 S', ''):
                row[11] = (stname.rsplit(" ", 1)[1].strip())
            else:
                row[11] = ""
            # set postal town with spatial join
            # set postal state
            row[12] = "NV"
            # set postal zip with spatial join
            # set owner first name
            ownfirst = row[13]
            if not (ownfirst is None or ownfirst.isspace() == True):
                row[14] = ownfirst
            else:
                row[14] = ""
            # set owner last name
            ownlast = row[15]
            if not (ownlast is None or ownlast.isspace() == True):
                row[16] = ownlast
            else:
                row[16] = ""
            #set owner full name
            if not (ownfirst is None and ownlast is None):
                row[17] = (ownfirst + " " + ownlast).strip()
            else:
                row[17] = ""
            # set mail1
            mail1 = row[18]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True or mail1 == "NONE" or mail1 == "NOT SUPPLIED"):
                row[19] = mail1
            else:
                row[19] = ""
            # set mial address 2
            mail2 = row[20]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and ("ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[21] = mail2
            else:
                row[21] = ""
            # set mail city
            mailcity = row[22]
            if not (mailcity is None or mailcity == "" or mailcity == "NONE" or mailcity.isspace() == True):
                row[23] = mailcity
            else:
                row[23] = ""
            # set mail state
            mailstate = row[24]
            if not (mailstate is None or mailstate == "" or mailstate.isspace() == True):
                row[25] = mailstate
            else:
                row[25] = ""
            # set mail zip
            mailzip = row[26]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[27] = mailzip[:5]
            else:
                row[27] = ""
            # set jurisdiction
            row[28] = County_ABR
            # set county
            row[29] = County_ABR
            # set ownership type
            if not (ownlast is None or ownlast == "" or ownlast.isspace() == True):
                if ownlast in fedOwnList:
                    row[30] = "Federal"
                elif ownlast in localOwnList:
                    row[30] = "Local"
                elif ownlast in stateOwnList:
                    row[30] = "State"
                elif not ownlast in (fedOwnList, localOwnList, stateOwnList):
                    row[30] = "Private"
            # set county land use code
            ctyluc = row[31]
            if not (ctyluc is None or ctyluc == "" or ctyluc.isspace() == True):
                row[32] = int(ctyluc)
            else:
                row[32] = ""
            # set assessed land value
            landval = row[33]
            row[34] = landval
            # set assessed improved value
            improval = row[35]
            row[36] = improval
            # set assessed sum
            if not (landval is None or improval is None):
                row[37] = landval + improval
            else:
                row[37] = ""
            # set tax land value
            taxlandval = row[38]
            if not taxlandval is None:
                row[39] = taxlandval
            else:
                row[39] = ""
            # set tax improved value
            taximproval = row[40]
            if not taximproval is None:
                row[41] = taximproval
            else:
                row[41] = ""
            # set tax sum
            if not (taxlandval is None or taximproval is None):
                row[42] = taxlandval + taximproval
            else:
                row[42] = ""
            # set tax year
            row[43] = current_year
            # update all rows
            cursor.updateRow(row)
        print ("Rows in {} county staging dataset have been updated".format(County_List[4]))       
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Carson":
    print ("Processing the {} dataset".format(County_List[0]))
    field_names = ['APN', 'APN_NEW', 'PPNO', 'PPNO_NEW','PLOC_','HSE_NUMBR_NEW','PLOCU_','UNIT_NUMBR_NEW', 'PLOCDR', 'STR_DIR_NEW',
                   'PLOCNM', 'STR_NAME_NEW', 'PLOCTP', 'STR_SUFFIX_NEW', 'PSTL_TOWN_NEW', 'PSTL_STATE_NEW','PSTL_ZIP5_NEW',
                   'PANAME', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW','PMADD1','MAIL_ADD1_NEW', 'PMADD2',
                   'MAIL_ADD2_NEW', 'PMCTST', 'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'PZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW',
                   'COUNTY', 'OWNERSHIP_TYPE', 'UNITS_NEW', 'YLDUSE', 'COUNTY_LANDUSE', 'YLANDV', 'AS_LANDVALUE_NEW',
                   'YIMPRV','AS_IMPROVALUE_NEW','AS_SUM_NEW','TAX_LANDVALUE_NEW','TAX_IMPROVALUE_NEW','TAX_SUM_NEW',
                   'TAX_YEAR_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set APN
            apn = row[0]
            row[1] = (apn[:3] + sep + apn[3:6] + sep + apn[6:8])
            # set PPNO
            row[2] = (apn)
            # set house number
            hsenum = row[3]
            if hsenum > 0:
                row[4] = hsenum
            else:
                row[4] = 0
            # set unit number
            row[5] = ""
            # set street direction
            stdir = row[34]
            if not stdir is None:
                row[63] = stdir
            else:
                row[63] = ""
            # set street name 
            stname = row[35]
            if not stname is None:
                row[64] = stname
            else:
                row[64] = ""
            # set street suffix
            row[65] = ""
            # APO Address set at the end
            # set postal town
            row[67] = 'Carson City'
            # set postal state
            row[68] = 'NV'
            # set postal zip
            row[69] = '89703' 
            # set owner first name
            own = row[27]
            if not (own is None or own == "" or own.isspace() == True):
                    try:
                        if own not in ownerListVal:
                            row[70] = (own.split(" ")[0].strip())
                    except IndexError:
                        row[70] = ""
            else:
                row[70] = ""
            # set owner last name
            if not (own is None or own == "" or own.isspace() == True):
                    try:
                        if own in ownerListVal:
                            row[71] = own
                        else:
                            row[71] = own.split(" ", 1)[1].strip()
                    except IndexError:
                        row[71] = ""
            else:
                row[71] = ""
            # set owner full name
            if not (own is None or own == "" or own.isspace() == True):
                row[72] = own
            else:
                row[72] = ""
            # set mail address 1
            mail1 = row[29]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True):
                row[73] = mail1.replace("%", "").strip()
            else:
                row[73] = ""
            # set mail address 2
            mail2 = row[30]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and (
            "ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[74] = mail2
            else:
                row[74] = ""
            # set mail city
            mailcity = row[31]
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True):
                row[75] = mailcity.split(",")[0].strip()
            else:
                row[75] = ""
            # set mail state
            mailstate = row[31]
            if "," in mailstate:
                mailstate = mailcity.split(",")[1].strip()
                row[76] = mailstate
            else:
                mailstate = ""
                row[76] = mailstate
            # set mail zip
            mailzip = row[32]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[77] = mailzip[:5]
            else:
                row[77] = ""
            # set jurisdiction
            row[78] = County_ABR
            #set county
            row[79] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[80] = "Federal"
                elif own in localOwnList:
                    row[80] = "Local"
                elif own in stateOwnList:
                    row[80] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[80] = "Private"
            # set county land use
            ctyluc = row[20]
            if ctyluc is not None:
                row[82] = ctyluc
            else:
                row[82] = ""
            # set assessed land value
            landval = row[42]
            if not landval is None:
                row[98] = landval
            else:
                row[98] = ""
            # set assessed improved value
            improval = row[43]
            if not improval is None:
                row[99] = improval
            else:
                row[99] = ""
            # set assessed sum
            row[118] = landval + improval
            # set tax land value
            taxland = landval / 0.35
            row[119] = taxland
            # set tax improved value
            taximprov = improval / 0.35
            row[120] = taximprov
            # set tax sum value
            row[121] = taxland + taximprov
            # set tax year
            row[122] = current_year
            
            # update all rows
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[0]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Douglas":
    print ("Processing the {} dataset".format(County_List[1]))
    field_names = ['APN', 'APN_NEW', 'PPNO', 'PPNO_NEW','PLOC_','HSE_NUMBR_NEW','PLOCU_','UNIT_NUMBR_NEW', 'PLOCDR', 'STR_DIR_NEW',
                   'PLOCNM', 'STR_NAME_NEW', 'PLOCTP', 'STR_SUFFIX_NEW', 'PSTL_TOWN_NEW', 'PSTL_STATE_NEW','PSTL_ZIP5_NEW',
                   'PANAME', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW','PMADD1','MAIL_ADD1_NEW', 'PMADD2',
                   'MAIL_ADD2_NEW', 'PMCTST', 'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'PZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW',
                   'COUNTY', 'OWNERSHIP_TYPE', 'UNITS_NEW', 'YLDUSE', 'COUNTY_LANDUSE_CODE', 'YLANDV', 'AS_LANDVALUE_NEW',
                   'YIMPRV','AS_IMPROVALUE_NEW','AS_SUM_NEW','TAX_LANDVALUE_NEW','TAX_IMPROVALUE_NEW','TAX_SUM_NEW',
                   'TAX_YEAR_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == ""):
                row[1] = (str(apn)[:4] + sep + str(apn)[4:6] + sep + str(apn)[6:9] + sep + str(apn)[9:12])
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not ppno is None:
                row[3] = ppno
            else:
                row[3] = 0
            # set house number
            hsenumbr = row[4]
            if hsenumbr > 0:
                row[5] = hsenumbr
            else:
                row[5] = 0
            # set unit number
            unit = row[6]
            if not unit is None:
                row[7] = "#" + unit
            else:
                row[7] = ""
            # set street direction
            stdir = row[8]
            if not stdir is None:
                row[9] = stdir
            else:
                row[9] = ""
            # set street name
            stname = row[10]
            if not stname is None:
                row[11] = stname
            else:
                row[11] = ""
            # set street suffix
            stsuf = row[12]
            if not stsuf is None:
                row[13] = stsuf
            else:
                row[13] = ""
            # set postal town
            row[14] = ""
            # set postal state
            row[15] = "NV"
            # set postal zip
            row[16] = ""
            # set owner fist name
            own = row[17]
            if not own is None or own == "" or own.isspace() == True:
                try:
                    if own not in ownerListVal:
                        row[18] = own.split(",", 1)[1].strip()
                    else:
                        row[18] = ""
                except IndexError:
                    row[18] = ""
            else:
                row[18] = ""
            # set owner last name
            if not own is None or own == "" or own.isspace() == True:
                try:
                    if own in ownerListVal:
                        row[19] = own
                    else:
                        row[19] = own.split(",")[0].strip()
                except IndexError:
                    row[19] = ""
            else:
                row[19] = ""
            # set owner fullname
            if not own is None:
                row[20] = own.strip()
            else:
                row[20] = ""
            
            # set mail address 1
            mail1 = row[23]
            if not mail1 is None or mail1 == "" or mail1.isspace() == True:
                row[22] = mail1.strip()
            else:
                row[22] = ""
            # set mail address 2
            mail2 = row[21]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) and (
                            "ATTN" in mail2 or "C/O" in mail2 or "PO BOX" in mail2 or "PMB" in mail2):
                row[24] = mail2.strip()
            else:
                row[24] = ""
            # set mail city
            mailcity = row[25]
            if not mailcity is None or mailcity == "" or mailcity.isspace() == True:
                if "," in mailcity:
                    row[26] = mailcity.split(",")[0].strip()
            else:
                row[26] = ""
            # set mail state
            if not mailcity is None or mailcity == "" or mailcity.isspace() == True:
                if "," in mailcity:
                    mailstate = mailcity.split(",")[1].strip()
                    if len(mailstate) > 2:
                        row[27] = ""
                        print ("International Address, {}".format(mailstate))
                    else:
                        row[27] = mailstate
            else:
                row[27] = ""
            # set mail zip code
            mailzip = row[28]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[29] = mailzip[:5]
            else:
                row[29] = ""
            # set jurisdiction
            row[30] = County_ABR
            # set county
            row[31] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[32] = "Federal"
                elif own in localOwnList:
                    row[32] = "Local"
                elif own in stateOwnList:
                    row[32] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[32] = "Private"
            # set units
            row[33] = 0
            # set county land use code
            ctyluc = row[34]
            row[35] = ctyluc
            # set assessed land value
            landval = row[36]
            if not landval is None:
                row[37] = landval
            else:
                row[37] = ""
            # set assessed improved value
            improval = row[38]
            if not improval is None:
                row[39] = improval
            else:
                row[39] = ""
            # set assessed sum
            row[40] = landval + improval
            # set tax land value
            taxland = landval / 0.35
            row[41] = taxland
            # set tax improved value
            taximprov = improval / 0.35
            row[42] = taximprov
            # set tax sum value
            row[43] = taxland + taximprov
            # set tax year
            row[44] = current_year
            
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[1]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "El_Dorado":
    print("Processing the {} dataset".format(County_List[2]))
    field_names = ['PRCL_ID', 'APN_NEW','PPNO_NEW','SITUSNUMBR','HSE_NUMBR_NEW','UNIT_NBR','UNIT_NUMBR_NEW', 
                   'STR_DIR_NEW','SITUSSTRNM', 'STR_NAME_NEW', 'SITUSSTRTY', 'STR_SUFFIX_NEW', 'PSTL_STATE_NEW',
                   'OWNER_NAME', 'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'OWNER_ADDR','MAIL_ADD1_NEW','MAIL_ADD2_NEW', 'OWNER_CITY', 'MAIL_CITY_NEW', 'OWNER_STAT','MAIL_STATE_NEW', 
                   'OWNER_ZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE',
                   'USECDPRI', 'COUNTY_LANDUSE_CODE',
                   'LANDVAL', 'AS_LANDVALUE_NEW','STRUCVAL','AS_IMPROVALUE_NEW','AS_SUM_NEW',
                   'TAX_LANDVALUE_NEW','TAX_IMPROVALUE_NEW','TAX_SUM_NEW','TAX_YEAR_NEW']
    # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn):
                row[1] = (apn[:3] + sep + apn[3:6] + sep + apn[6:8])
            else:
                row[1] = ""
            # set ppno
            ppno = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or 'UN' in apn):
                try:
                    row[2] =  ppno
                except ValueError:
                    row[2] = ""
            else:
                row[2] = ""
             # set house number
            hsenum = row[3]
            row[4] = hsenum
            # set unit number
            unit = row[5]
            if not unit is None:
                row[6] = "#" + unit
            else:
                row[6] = ""
            # set street direction doesnt exist in El Dorado data
            row[7] = ""
            # set street name
            stname = row[8]
            if not stname is None:
                row[9] = stname
            else:
                row[9] = ""
            # set street suffix
            stsuff = row[10]
            if not stsuff is None:
                row[11] = stsuff
            else:
                row[11] = ""
            # set postal town and postal zip with spatial join....later in the python notebook....scroll down
            # set postal state
            row[12] = "CA"
            # set owner full name
            own = row[13]
            if not own is None:
                row[16] = own
            else:
                row[16] = ""
            # set owner first name
            if not own is None or own == "" or own.isspace() == True:
                try:
                    if own not in ownerListVal:
                        row[14] = own.split(" ", 1)[1].strip()
                except IndexError:
                    row[14] = ""
            else:
                row[14] = ""
            # set owner last name
            if not own is None or own == "" or own.isspace() == True:
                try:
                    if own in ownerListVal:
                        row[15] = own
                    else:
                        row[15] = own.split(" ")[0].strip()
                except IndexError:
                    row[15] = ""
            else:
                row[15] = ""
            # set mail1
            mail1 = row[17]
            if not mail1 is None or mail1 == "" or mail1.isspace() == True:
                row[18] = mail1.replace("%", "").strip()
            else:
                row[18] = ""
            # set mail2
            row[19] = ""
            # set mail city
            mailcity = row[20]
            if not mailcity is None or mailcity == "" or mailcity.isspace() == True:
                row[21] = mailcity
            else:
                row[21] = ""
            # set mail state
            mailstate = row[22]
            if not mailstate is None or mailstate == "" or mailstate.isspace() == True:
                row[23] = mailstate
            else:
                row[23] = ""
            # set mail zip
            mailzip = row[24]
            if not mailzip is None or mailzip == "" or mailzip.isspace() == True:
                row[25] = mailzip[:5]
            else:
                row[25] = ""
            # set jurisdiction
            row[26] = County_ABR
            # set county
            row[27] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[28] = "Federal"
                elif own in localOwnList:
                    row[28] = "Local"
                elif own in stateOwnList:
                    row[28] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[28] = "Private"
            # set county land use code
            ctyluc = row[29]
            if not ctyluc is None or ctyluc == "" or ctyluc.isspace() == True:
                row[30] = ctyluc
            else:
                row[30] = ""
            # set assessed land value
            landval = row[31]
            row[32] = landval
            # set assessed improved value
            improval = row[33]
            row[34] = improval
            # set assessed sum
            row[35] = improval + landval
            # set tax land value
            taxland = row[31]
            row[36] = taxland
            # set tax improved value
            taximprov = row[33]
            row[37] = taximprov
            # set tax sum
            row[38] = taxland + taximprov
            # set tax year
            row[39] = current_year

            # set postl zip with spatial join
            # set number of units
            
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[2]))
# ----------------------------------------------------------------------------------------------
elif UserCountyResponse == "Placer":
    print ("Processing the {} dataset".format(County_List[3]))
    field_names = ['APN', 'APN_NEW','GISAPN', 'PPNO_NEW','STREETNUM','HSE_NUMBR_NEW','SP_APT','UNIT_NUMBR_NEW', 
                   'STREETDIR', 'STR_DIR_NEW','STREETNAME', 'STR_NAME_NEW', 'STREETTYPE', 'STR_SUFFIX_NEW', 'PSTL_STATE_NEW',
                   'OWNER1', 'OWN_FIRST_NEW', 'OWNER2', 'OWN_LAST_NEW', 'OWN_FULL_NEW',
                   'ADR1','MAIL_ADD1_NEW', 'MAIL_ADD2_NEW','CITY', 'MAIL_CITY_NEW', 'STATE','MAIL_STATE_NEW', 
                   'ZIP',  'MAIL_ZIP5_NEW', 'JURISDICTION_NEW','COUNTY', 'OWNERSHIP_TYPE',
                   'USE_CD', 'COUNTY_LANDUSE_CODE', 'USE_CD_N', 'COUNTY_LANDUSE_DESCRIPTION',
                   'LANDVALUE', 'AS_LANDVALUE_NEW','STRUCTURE','AS_IMPROVALUE_NEW','AS_SUM_NEW',
                   'TAX_LANDVALUE_NEW','TAX_IMPROVALUE_NEW','TAX_SUM_NEW','TAX_YEAR_NEW']
   # lists the index of the field names in parcel out data
    for index, field in enumerate(field_names):
        print (index, field)
    with arcpy.da.UpdateCursor(parcel_out_data, field_names) as cursor:
        for row in cursor:
            # set apn
            apn = row[0]
            if not (apn is None or apn == "" or apn.isspace() == True or "ROW" in apn or len(apn) < 8):
                row[1] = apn[:11]
            else:
                row[1] = ""
            # set ppno
            ppno = row[2]
            if not (ppno is None or ppno == "" or ppno.isspace() == True or "ROW" in ppno or len(ppno) < 8):
                row[3] =  int(ppno)
            else:
                row[3] = 0
            # set house number
            hsenumbr = row[4]
            if not (hsenumbr is None or hsenumbr == "" or hsenumbr.isspace() == True):
                try:
                    if hsenumbr.isdigit():
                        row[5] = hsenumbr
                    elif "'" in hsenumbr:
                        row[5] = hsenumbr.replace("'", "").strip()
                    else:
                        pass
                except RuntimeError:
                    row[5] = 0
            else:
                row[5] = 0
            # set unit number
            unit = row[6]
            if not unit is None:
                row[7] = "#" + unit
            else:
                row[7] = ""
            # set street direction
            stdir = row[8]
            if not stdir is None:
                row[9] = stdir
            else:
                row[9] = ""
            # set street name
            stname = row[10]
            if not stname is None:
                row[11] = stname
            else:
                row[11] = ""
            # set street suffix
            stsuff = row[12]
            if not stsuff is None:
                row[13] = stsuff
            else:
                row[13] = ""
            # set postal state
            row[14] =  "CA"
            
            # set postal town and postal zip with spatial join....scroll down
            # set owner full name
            own = row[15]
            if not own is None:
                row[19] = own
            else:
                row[19] = ""
            # set owner first name
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own not in ownerListVal:
                        row[16] = own.split(" ", 1)[1].strip()
                except IndexError:
                    row[16] = ""
            else:
                row[16] = ""
            # set owner last name
            if not (own is None or own == "" or own.isspace() == True):
                try:
                    if own in ownerListVal:
                        row[17] = own
                    else:
                        row[17] = own.split(" ")[0].strip()
                except IndexError:
                    row[17] = ""
            else:
                row[17] = ""
            # set mail1
            mail1 = row[20]
            if not (mail1 is None or mail1 == "" or mail1.isspace() == True):
                row[21] = mail1
            else:
                row[20] = ""
            # set mail2
            mail2 = row[21]
            if not (mail2 is None or mail2 == "" or mail2.isspace() == True) \
                    and ("ATTN" in mail2 or "C/O" in mail2 or "P O BOX" in mail2 or "PMB" in mail2):
                row[22] = mail2
            else:
                row[22] = ""
            # set mail city
            mailcity = row[23]
            if not (mailcity is None or mailcity == "" or mailcity.isspace() == True or "N/A" in mailcity or \
                    "N\A" in mailcity or "P O BOX" in mailcity or "VALUE" in mailcity or "PRORATED" in mailcity):
                row[24] = mailcity
            else:
                row[24] = ""
            # set mail state
            mailstate = row[25]
            if not (mailstate is None or mailstate == "" or mailstate.isspace() == True or len(mailstate) < 2):
                row[26] = mailstate
            else:
                row[26] = ""
            # set mail zip
            mailzip = row[27]
            if not (mailzip is None or mailzip == "" or mailzip.isspace() == True):
                row[28] = mailzip[:5]
            else:
                row[28] = ""
            # set jurisdiction
            row[29] = County_ABR
            # set county
            row[30] = County_ABR
            # set ownership type
            if not (own is None or own == "" or own.isspace() == True):
                if own in fedOwnList:
                    row[31] = "Federal"
                elif own in localOwnList:
                    row[31] = "Local"
                elif own in stateOwnList:
                    row[31] = "State"
                elif not own in (fedOwnList, localOwnList, stateOwnList):
                    row[31] = "Private"
            # set county land use code
            ctyluc = row[32]
            if not (ctyluc is None or ctyluc == "" or ctyluc.isspace() == True):
                row[33] = ctyluc
            else:
                row[33] = ""
            # set assessed land value
            landval = row[36]
            row[37] = landval
            # set assessed improved value
            improval = row[38]
            row[39] = improval
            # set assessed sum
            if not (landval is None and improval is None):
                row[40] =  landval + improval
            else:
                row[40] = ""
            # set tax land value
            if not (landval is None):
                row[41] =  landval
            else:
                row[41] = ""
            # set tax impoved value
            if not (improval is None):
                row[42] =  improval
            else:
                row[42] = ""
            # set tax sum
            if not (landval is None and improval is None):
                row[43] =  landval + improval
            else:
                row[43] = ""
            # set tax year
            row[44] = current_year
            
            cursor.updateRow(row)
        print ("Rows in {} dataset have been updated".format(County_List[3]))
#----------------------------------------------------------------------------------------            
# Calculate APO_ADDRESS, PARCEL_ACRES, and PARCEL_SQFT
expression = "concat(!HSE_NUMBR_NEW!, !STR_DIR_NEW!, !STR_NAME_NEW!, !STR_SUFFIX_NEW!, !UNIT_NUMBR_NEW!)"
apo_codeblock = """
def concat(*args):
    retval = ""
    sep = " "
    for t in args:
        s = str(t).strip()
        if s != None:
            retval += sep + s
    return retval.lstrip(sep)"""
arcpy.CalculateField_management(parcel_out_data, "APO_ADDRESS_NEW", expression, python_version, apo_codeblock)
print ("Calculated APO_ADDRESS")
arcpy.CalculateField_management(parcel_out_data, "PARCEL_ACRES_NEW", "!shape.area@acres!", python_version, "")
print ("Calculated Parcel Acreage")
arcpy.CalculateField_management(parcel_out_data, "PARCEL_SQFT_NEW", "!shape.area@squarefeet!", python_version, "")
print ("Calculated Parcel SQFT")    
arcpy.CalculateField_management(parcel_out_data, "APO_ADDRESS_NEW", "' '.join(!APO_ADDRESS_NEW!.strip().split())", python_version,"#")
print ("Removed Double Spaces from APO_ADDRESS")

Processing the Washoe dataset
0 PIN
1 APN_NEW
2 APN
3 PPNO_NEW
4 STREETNUM
5 HSE_NUMBR_NEW
6 UNIT_NUMBR_NEW
7 STREETDIR
8 STR_DIR_NEW
9 STREET
10 STR_NAME_NEW
11 STR_SUFFIX_NEW
12 PSTL_STATE_NEW
13 FIRSTNAME
14 OWN_FIRST_NEW
15 LASTNAME
16 OWN_LAST_NEW
17 OWN_FULL_NEW
18 MAILING1
19 MAIL_ADD1_NEW
20 MAILING2
21 MAIL_ADD2_NEW
22 MAILCITY
23 MAIL_CITY_NEW
24 MAILSTATE
25 MAIL_STATE_NEW
26 MAILZIP
27 MAIL_ZIP5_NEW
28 JURISDICTION_NEW
29 COUNTY
30 OWNERSHIP_TYPE
31 LAND_USE
32 COUNTY_LANDUSE_CODE
33 LANDASS
34 AS_LANDVALUE_NEW
35 BUILDASS
36 AS_IMPROVALUE_NEW
37 AS_SUM_NEW
38 LANDAPR
39 TAX_LANDVALUE_NEW
40 BUILDAPR
41 TAX_IMPROVALUE_NEW
42 TAX_SUM_NEW
43 TAX_YEAR_NEW
Rows in Washoe county staging dataset have been updated
Calculated APO_ADDRESS
Calculated Parcel Acreage
Calculated Parcel SQFT
Removed Double Spaces from APO_ADDRESS


### Merge Staging Datasets and Eliminate County Fields

In [42]:
# Parcel staging feature classes to be merged
staging = "F:\\GIS\\ParcelUpdate\\2019_04"
WA_Staging = staging + "\\Washoe\\Modified\\WA_Staging.gdb\\Parcels\\Washoe_Parcels"
#CC_Staging = staging + "\\Carson\\Modified\\CC_Staging.gdb\\Parcels\\Carson_Parcels"
DG_Staging = staging + "\\Douglas\\Modified\\DG_Staging.gdb\\Parcels\\Douglas_Parcels"
EL_Staging = staging + "\\El_Dorado\Modified\EL_Staging.gdb\Parcels\\El_Dorado_Parcels"
PL_Staging = staging + "\\Placer\\Modified\\PL_Staging.gdb\\Parcels\\Placer_Parcels"

# set output staging feature class
Parcel_Master_New = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_New"

# Create FieldMappings object to manage merge output fields
fieldMappings = arcpy.FieldMappings()

# Add all fields
fieldMappings.addTable(WA_Staging)
#fieldMappings.addTable(CC_Staging)
fieldMappings.addTable(DG_Staging)
fieldMappings.addTable(EL_Staging)
fieldMappings.addTable(PL_Staging)

# Remove all output fields from the field mappings, except fields in field_master list
for field in fieldMappings.fields:
    if field.name not in ['APN_NEW', 'PPNO_NEW', 'HSE_NUMBR_NEW', 'UNIT_NUMBR_NEW', 'STR_DIR_NEW', 'STR_NAME_NEW', 
                'STR_SUFFIX_NEW', 'APO_ADDRESS_NEW', 'PSTL_TOWN_NEW', 'PSTL_STATE_NEW', 'PSTL_ZIP5_NEW', 
                'OWN_FIRST_NEW', 'OWN_LAST_NEW', 'OWN_FULL_NEW', 'MAIL_ADD1_NEW', 'MAIL_ADD2_NEW', 
                'MAIL_CITY_NEW', 'MAIL_STATE_NEW', 'MAIL_ZIP5_NEW', 'JURISDICTION_NEW', 'COUNTY', 'OWNERSHIP_TYPE', 
                'UNITS_NEW', 'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
                'REGIONAL_LANDUSE', 'SOIL_1974', 'SOIL_2003', 'ALLOWABLE_COVERAGE_BAILEY_SQFT', 
                'IMPERVIOUS_SURFACE_SQFT', 'HRA_NAME', 'WATERSHED_NUMBER', 'WATERSHED_NAME', 'PRIORITY_WATERSHED',  
                'BMP_STATUS', 'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 'AS_LANDVALUE_NEW', 'AS_IMPROVALUE_NEW', 
                'AS_SUM_NEW', 'TAX_LANDVALUE_NEW', 'TAX_IMPROVALUE_NEW', 'TAX_SUM_NEW', 'TAX_YEAR_NEW', 'PAS_ID', 
                'PAS_NAME', 'INDEX_1987', 'INDEX_1987_HYPERLINK', 'LOCAL_PLAN_HYPERLINK', 'PARCEL_DETAILS_HYPERLINK' 'ZONING_NEW', 
                'ZONING_DESCRIPTION', 'SINGLE_FAMILY_DENSITY', 'MULTI_FAMILY_DENSITY', 
                'TOURIST_ACCOMMODATION_DENSITY', 'BED_BREAKFAST_DENSITY', 'TIME_SHARE_DENSITY', 
                'COMMERCIAL_FLOOR_AREA_ALLOWED', 'SECONDARY_DWELLING_UNIT_ALLOWED', 'OVERLAY', 
                'MAX_CP_RES_DENSITY', 'MAX_CP_TAU_DENSITY', 'SPECIAL_PLAN_AREA_OVERLAY', 'TOWN_CENTER', 
                'SPECIAL_AREA', 'SPECIAL_PLANNING_DISTRICT', 'SPECIAL_PLANNING_DISTRICT_TYPE', 
                'LOCATION_TO_TOWNCENTER', 'RETIRED', 'PARCEL_ACRES_NEW', 'PARCEL_SQFT_NEW', 'DUPLICATE']:
        fieldMappings.removeFieldMap(fieldMappings.findFieldMapIndex(field.name))
for field in fieldMappings.fields:
    print ("Created Field Map:", field.name)

# Use Merge tool to move features into single dataset
arcpy.Merge_management([WA_Staging, DG_Staging, EL_Staging, PL_Staging], Parcel_Master_New, fieldMappings)
print ("Created a new parcel dataset")

Created Field Map: APN_NEW
Created Field Map: PPNO_NEW
Created Field Map: HSE_NUMBR_NEW
Created Field Map: UNIT_NUMBR_NEW
Created Field Map: STR_DIR_NEW
Created Field Map: STR_NAME_NEW
Created Field Map: STR_SUFFIX_NEW
Created Field Map: APO_ADDRESS_NEW
Created Field Map: PSTL_TOWN_NEW
Created Field Map: PSTL_STATE_NEW
Created Field Map: PSTL_ZIP5_NEW
Created Field Map: OWN_FIRST_NEW
Created Field Map: OWN_LAST_NEW
Created Field Map: OWN_FULL_NEW
Created Field Map: MAIL_ADD1_NEW
Created Field Map: MAIL_ADD2_NEW
Created Field Map: MAIL_CITY_NEW
Created Field Map: MAIL_STATE_NEW
Created Field Map: MAIL_ZIP5_NEW
Created Field Map: JURISDICTION_NEW
Created Field Map: COUNTY
Created Field Map: OWNERSHIP_TYPE
Created Field Map: UNITS_NEW
Created Field Map: COUNTY_LANDUSE_CODE
Created Field Map: COUNTY_LANDUSE_DESCRIPTION
Created Field Map: TRPA_LANDUSE_DESCRIPTION
Created Field Map: REGIONAL_LANDUSE
Created Field Map: SOIL_1974
Created Field Map: SOIL_2003
Created Field Map: ALLOWABLE_COVERA

### Change Field Names (eliminate the '_NEW')

In [43]:
import os
import arcpy

fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master_New" 
new_fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master" 

field_mappings = arcpy.FieldMappings() # Create new field mapping object

#Loop through fields. For each field, create a corresponding FieldMap object.
#If the name ends with "_NEW", strip it off.
#Note: "FID and "Shape" are skipped

for field in arcpy.ListFields(fc):
    if not field.name == "OBJECTID" and not field.name == "Shape":
        old_name = field.name

        #Rename if necessary
        if old_name.endswith("_NEW"):
            new_name = old_name[:-4]
        else:
            new_name = old_name

        #Create new FieldMap object    
        new_f = arcpy.FieldMap()
        new_f.addInputField(fc, old_name) # Specify the input field to use

        #Rename output field
        new_f_name = new_f.outputField
        new_f_name.name = new_name
        new_f_name.aliasName = new_name
        new_f.outputField = new_f_name

        #Add field to FieldMappings object
        field_mappings.addFieldMap(new_f)

#Convert table using your created Field Mappings object
arcpy.FeatureClassToFeatureClass_conversion(fc, os.path.dirname(new_fc), os.path.basename(new_fc), field_mapping=field_mappings)

<Result 'F:\\GIS\\ParcelUpdate\\2019_04\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master'>

## Update Field Values

### Field Calculator Function

In [4]:
import arcpy

def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    
    #updateFC = r"C:\Path\UpdateFeatureClass"  
    #updateFieldsList = ["JoinField", "ValueField"]
    
    # sourceFC = r"C:\Path\SourceFeatureClass"  
    #sourceFieldsList = ["JoinField", "ValueField"]  
    
    # Use list comprehension to build a dictionary from a da SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

### Set local variables again...

In [5]:
import arcpy
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
ParcelPoint = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Point"

#ParcelLayer = "C:\\GIS\\PROJECT\\2018_10\\Workspace\\ParcelUpdate_Workspace.gdb\\Parcel_Master"
#ParcelPoint = "C:\\GIS\\PROJECT\\2018_10\\Workspace\\ParcelUpdate_Workspace.gdb\\Parcel_Point"

if arcpy.Exists(ParcelPoint):
    arcpy.Delete_management(ParcelPoint)
arcpy.FeatureToPoint_management(ParcelLayer, ParcelPoint, "INSIDE")

# SDE feature classes from C: drive
# sde_FireDistrict = "C:\\GIS\\PROJECT\\SDE\\SDE_Archive_2018.gdb\\Jurisdictions\\FireDistricts"

# File paths
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"
#sde feature classes
sde_FireDistrict = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.FireDistricts"
sde_NRCSSoils1974 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_1974"
sde_NRCSSoils2003 = sdeBase + "\\sde.SDE.Soils\\sde.SDE.NRCS_Soils_2003"
sde_HydroArea = sdeBase + "\\sde.SDE.Water\\sde.SDE.Hydro_Areas"
sde_Watershed = sdeBase + "\\sde.SDE.Water\\sde.SDE.Priority"
sde_RegionalLandUse = sdeBase + "\\sde.SDE.Planning\\sde.SDE.RegionalLandUse"
sde_LocalPlan = sdeBase + "\\sde.SDE.Planning\\sde.SDE.LocalPlan"
sde_Zoning =  sdeBase + "\\sde.SDE.Planning\\sde.SDE.Zoning_LocalPlan"
sde_SpecialDistrict = sdeBase + "\\sde.SDE.Planning\\sde.SDE.SpecialPlanningDistrict"
sde_TownCenter = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter"
sde_TownCenterBuffer = sdeBase + "\\sde.SDE.Planning\\sde.SDE.TownCenter_Buffer"
sde_Index1987 = sdeBase + "\\sde.SDE.Planning\\sde.SDE.AssessorMapIndex_1987"
sde_TRPAboundary = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.TRPA_bdy"
sde_UrbanArea = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.UrbanAreas"
sde_Zip = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_Littoral = sdeBase + "\\sde.SDE.Census\\sde.SDE.Tahoe_Census_Zip"
sde_CSLT = sdeBase + "\\sde.SDE.Jurisdictions\\sde.SDE.CSLT"

# in memory files
wk_memory = "in_memory" + "\\"
ParcelPoint_FireDistrict = wk_memory + "ParcelPoint_FireDistrict"
ParcelPoint_Soils74 = wk_memory + "\\ParcelPoint_Soils74"
ParcelPoint_Soils03 = wk_memory + "\\ParcelPoint_Soils03"
ParcelPoint_HydroArea = wk_memory + "\\ParcelPoint_HydroArea"
ParcelPoint_Watershed = wk_memory + "\\ParcelPoint_Watershed"
ParcelPoint_RegionalLandUse = wk_memory + "\\ParcelPoint_RegionalLandUse"
ParcelPoint_LocalPlan = wk_memory + "\\ParcelPoint_LocalPlan"
ParcelPoint_TownCen6ter = wk_memory + "\\ParcelPoint_TownCenter"
ParcelPoint_TownCenterBuffer = wk_memory + "\\ParcelPoint_TownCenterBuffer"
# ParcelPoint_Zoning = wk_memory + "\\ParcelPoint_Zoning"
ParcelPoint_Zoning = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\ParcelPoint_Zoning"
ParcelPoint_SpecialDistrict = wk_memory + "\\ParcelPoint_SpecialDistrict"
ParcelPoint_Index1987 = wk_memory + "\\ParcelPoint_Index1987"
ParcelPoint_PstlTown = wk_memory + "\\ParcelPoint_PstlTown"
ParcelPoint_PstlZip = wk_memory + "\\ParcelPoint_PstlZip"
ParcelPoint_TRPAboundary= wk_memory + "\\ParcelPoint_TRPABoundary"
ParcelPoint_CSLT = wk_memory + "\\ParcelPoint_CSLT"

### Update Fire Protection District

In [7]:
# delete feature class if it exists already
if arcpy.Exists(ParcelPoint_FireDistrict):
    arcpy.Delete_management(ParcelPoint_FireDistrict)

print("Starting the Fire District Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_FireDistrict, ParcelPoint_FireDistrict, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Fire District Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'FIREPD'], ParcelPoint_FireDistrict, ['APN', 'DISTRICT'])
print ("The 'FIREPD' field in the parcel data has been updated")

Starting the Fire District Spatial Join
Finished the Fire District Spatial Join
Started data transfer: 2019-04-25 12:14:14
Finished data transfer: 2019-04-25 12:14:16
The 'FIREPD' field in the parcel data has been updated


### Update Soils 1974 Field

In [8]:
# delete feature class if it exists already
if arcpy.Exists(ParcelPoint_Soils74):
    arcpy.Delete_management(ParcelPoint_Soils74)

print("Starting the SOIL_1974 Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils1974, ParcelPoint_Soils74, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_1974 Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SOIL_1974'], ParcelPoint_Soils74, ['APN', 'MUSYM_74'])
print ("The 'SOIL_1974' field in the parcel data has been updated")

Starting the SOIL_1974 Spatial Join
Finished the SOIL_1974 Spatial Join
Started data transfer: 2019-04-25 12:14:33
Finished data transfer: 2019-04-25 12:14:35
The 'SOIL_1974' field in the parcel data has been updated


### Update Soils 2003 Field

In [9]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Soils03):
    arcpy.Delete_management(ParcelPoint_Soils03)

print("Starting the SOIL_2003 Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_NRCSSoils2003, ParcelPoint_Soils03, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the SOIL_2003 Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SOIL_2003'], ParcelPoint_Soils03, ['APN', 'MUSYM_03'])
print ("The 'SOIL_2003' field in the parcel data has been updated")

Starting the SOIL_2003 Spatial Join
Finished the SOIL_2003 Spatial Join
Started data transfer: 2019-04-25 13:08:10
Finished data transfer: 2019-04-25 13:08:12
The 'SOIL_2003' field in the parcel data has been updated


### Update Hydrologic Area Field

#### Pseudo Code
* Make sure this runs for all Parcels! Last time it only updated a few of the fields values

In [5]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_HydroArea):
    arcpy.Delete_management(ParcelPoint_HydroArea)

print("Starting the Hyrdrologic Area Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_HydroArea, ParcelPoint_HydroArea, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Hydrologic Area Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'HRA_NAME'], ParcelPoint_HydroArea, ['APN', 'HRA_NAME'])
print ("The 'HRA_NAME' field in the parcel data has been updated")

Starting the Hyrdrologic Area Spatial Join
Finished the Hydrologic Area Spatial Join
Started data transfer: 2019-05-08 09:52:26
Finished data transfer: 2019-05-08 09:52:28
The 'HRA_NAME' field in the parcel data has been updated


### Update Watershed Fields

In [11]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Watershed):
    arcpy.Delete_management(ParcelPoint_Watershed)

print("Starting the Watershed Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Watershed, ParcelPoint_Watershed, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Watershed Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'WATERSHED_NUMBER'], ParcelPoint_Watershed, ['APN', 'NUMBER'])
print ("The 'WATERSHED_NUMBER' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'WATERSHED_NAME'], ParcelPoint_Watershed, ['APN', 'NAME'])
print ("The 'WATERSHED_NAME' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'PRIORITY_WATERSHED'], ParcelPoint_Watershed, ['APN', 'PRIORITY'])
print ("The 'PRIORITY_WATERSHED' field in the parcel data has been updated")

Starting the Watershed Spatial Join
Finished the Watershed Spatial Join
Started data transfer: 2019-04-25 13:09:04
Finished data transfer: 2019-04-25 13:09:06
The 'WATERSHED_NUMBER' field in the parcel data has been updated
Started data transfer: 2019-04-25 13:09:06
Finished data transfer: 2019-04-25 13:09:08
The 'WATERSHED_NAME' field in the parcel data has been updated
Started data transfer: 2019-04-25 13:09:08
Finished data transfer: 2019-04-25 13:09:10
The 'PRIORITY_WATERSHED' field in the parcel data has been updated


### Update Regional Land Use Field

In [12]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_RegionalLandUse):
    arcpy.Delete_management(ParcelPoint_RegionalLandUse)

print("Starting the Regional Land Use Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_RegionalLandUse, ParcelPoint_RegionalLandUse, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Regional Land Use Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'REGIONAL_LANDUSE'], ParcelPoint_RegionalLandUse, ['APN', 'Description'])
print ("The 'REGIONAL_LANDUSE' field in the parcel data has been updated")

Starting the Regional Land Use Spatial Join
Finished the Regional Land Use Spatial Join
Started data transfer: 2019-04-25 13:09:42
Finished data transfer: 2019-04-25 13:09:44
The 'REGIONAL_LANDUSE' field in the parcel data has been updated


### Update Local Plan Fields

In [4]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_LocalPlan):
    arcpy.Delete_management(ParcelPoint_LocalPlan)

print("Starting the Local Plan Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_LocalPlan, ParcelPoint_LocalPlan, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Local Plan Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PAS_ID'], ParcelPoint_LocalPlan, ['APN', 'PLAN_ID'])
print ("The 'PAS_ID' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'PAS_NAME'], ParcelPoint_LocalPlan, ['APN', 'PLAN_NAME'])
print ("The 'PAS_NAME' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'LOCAL_PLAN_HYPERLINK'], ParcelPoint_LocalPlan, ['APN', 'File_URL'])
print ("The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated")

Starting the Local Plan Spatial Join
Finished the Local Plan Spatial Join
Started data transfer: 2019-06-25 15:28:06
Finished data transfer: 2019-06-25 15:28:10
The 'PAS_ID' field in the parcel data has been updated
Started data transfer: 2019-06-25 15:28:11
Finished data transfer: 2019-06-25 15:28:14
The 'PAS_NAME' field in the parcel data has been updated
Started data transfer: 2019-06-25 15:28:15
Finished data transfer: 2019-06-25 15:28:18
The 'LOCAL_PLAN_HYPERLINK' field in the parcel data has been updated


### Update Town Center Field

In [8]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TownCenter):
    arcpy.Delete_management(ParcelPoint_TownCenter)

print("Starting the Town Center Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenter, ParcelPoint_TownCenter, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'TOWN_CENTER'], ParcelPoint_TownCenter, ['APN', 'NAME'])
print ("The 'TOWN_CENTER' field in the parcel data has been updated")

Starting the Town Center Spatial Join
Finished the Town Center Spatial Join
Started data transfer: 2019-06-25 16:38:16
Finished data transfer: 2019-06-25 16:38:19
The 'TOWN_CENTER' field in the parcel data has been updated


### Update Town Center Buffer Field

In [6]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TownCenterBuffer):
    arcpy.Delete_management(ParcelPoint_TownCenterBuffer)

print("Starting the Town Center Buffer Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TownCenterBuffer, ParcelPoint_TownCenterBuffer, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Town Center Buffer Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'LOCATION_TO_TOWNCENTER'], ParcelPoint_TownCenterBuffer, ['APN', 'Category'])
print ("The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated")

Starting the Town Center Buffer Spatial Join
Finished the Town Center Buffer Spatial Join
Started data transfer: 2019-06-25 15:30:07
Finished data transfer: 2019-06-25 15:30:12
The 'LOCATION_TO_TOWNCENTER' field in the parcel data has been updated


### Update Zoning Fields

In [7]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Zoning):
    arcpy.Delete_management(ParcelPoint_Zoning)

print("Starting the Zoning Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zoning, ParcelPoint_Zoning, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Zoning Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'ZONING'], ParcelPoint_Zoning, ['APN', 'Zoning_1'])
print ("The 'ZONING' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'ZONING_DESCRIPTION'], ParcelPoint_Zoning, ['APN', 'Zoning_Description_1'])
print ("The 'ZONING_DESCRIPTION' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'SINGLE_FAMILY_DENSITY'], ParcelPoint_Zoning, ['APN', 'Single_Family_Density_1'])
print ("The 'SINGLE_FAMILY_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'MULTI_FAMILY_DENSITY'], ParcelPoint_Zoning, ['APN', 'Multi_Family_Density_1'])
print ("The 'MULTI_FAMILY_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'TOURIST_ACCOMMODATION_DENSITY'], ParcelPoint_Zoning, ['APN', 'Tourist_Accommodation_Density_1'])
print ("The 'TOURIST_ACCOMMODATION_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'BED_BREAKFAST_DENSITY'], ParcelPoint_Zoning, ['APN', 'Bed_Breakfast_Density_1'])
print ("The 'BED_BREAKFAST_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'TIME_SHARE_DENSITY'], ParcelPoint_Zoning, ['APN', 'Time_Share_Density_1'])
print ("The 'TIME_SHARE_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'COMMERCIAL_FLOOR_AREA_ALLOWED'], ParcelPoint_Zoning, ['APN', 'Commercial_Floor_Area'])
print ("The 'COMMERCIAL_FLOOR_AREA_ALLOWED' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'SECONDARY_DWELLING_UNIT_ALLOWED'], ParcelPoint_Zoning, ['APN', 'Secondary_Dwelling_Unit'])
print ("The 'SECONDARY_DWELLING_UNIT_ALLOWED' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'OVERLAY'], ParcelPoint_Zoning, ['APN', 'Overlay_1'])
print ("The 'OVERLAY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'MAX_CP_RES_DENSITY'], ParcelPoint_Zoning, ['APN', 'Max_Community_Plan_Res_Density'])
print ("The 'MAX_CP_RES_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'MAX_CP_TAU_DENSITY'], ParcelPoint_Zoning, ['APN', 'Max_CP_Tourist_Accom_Den'])
print ("The 'MAX_CP_TAU_DENSITY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'SPECIAL_PLAN_AREA_OVERLAY'], ParcelPoint_Zoning, ['APN', 'Special_Planning_Area_Overlay'])
print ("The 'SPECIAL_PLAN_AREA_OVERLAY' field in the parcel data has been updated")

fieldJoinCalc(ParcelLayer, ['APN', 'DesignGuidelines_URL'], ParcelPoint_Zoning, ['APN', 'DesignGuidelines_URL_1'])
print ("The 'Design Guidelines URL' field in the parcel data has been updated")

Starting the Zoning Spatial Join
Finished the Zoning Spatial Join
Started data transfer: 2019-06-25 16:34:23
Finished data transfer: 2019-06-25 16:34:35
The 'ZONING' field in the parcel data has been updated
Started data transfer: 2019-06-25 16:34:37
Finished data transfer: 2019-06-25 16:34:42
The 'ZONING_DESCRIPTION' field in the parcel data has been updated
Started data transfer: 2019-06-25 16:34:44
Finished data transfer: 2019-06-25 16:34:47
The 'SINGLE_FAMILY_DENSITY' field in the parcel data has been updated
Started data transfer: 2019-06-25 16:34:50
Finished data transfer: 2019-06-25 16:34:53
The 'MULTI_FAMILY_DENSITY' field in the parcel data has been updated
Started data transfer: 2019-06-25 16:34:56
Finished data transfer: 2019-06-25 16:35:00
The 'TOURIST_ACCOMMODATION_DENSITY' field in the parcel data has been updated
Started data transfer: 2019-06-25 16:35:04
Finished data transfer: 2019-06-25 16:35:08
The 'BED_BREAKFAST_DENSITY' field in the parcel data has been updated
Sta

### Update Special Area Fields

In [6]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_SpecialDistrict):
    arcpy.Delete_management(ParcelPoint_SpecialDistrict)

print("Starting the Special Planning District Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_SpecialDistrict, ParcelPoint_SpecialDistrict, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Special Planning District Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SPECIAL_PLANNING_DISTRICT'], ParcelPoint_SpecialDistrict, ['APN', 'NAME'])
print ("The 'SPECIAL_PLANNING_DISTRICT' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'SPECIAL_PLANNING_DISTRICT_TYPE'], ParcelPoint_SpecialDistrict, ['APN', 'TYPE'])
print ("The 'SPECIAL_PLANNING_DISTRICT_TYPE' field in the parcel data has been updated")

Starting the Special Planning District Spatial Join
Finished the Special Planning District Spatial Join
Started data transfer: 2019-06-25 12:12:12
Finished data transfer: 2019-06-25 12:12:17
The 'SPECIAL_PLANNING_DISTRICT' field in the parcel data has been updated
Started data transfer: 2019-06-25 12:12:18
Finished data transfer: 2019-06-25 12:12:21
The 'SPECIAL_PLANNING_DISTRICT_TYPE' field in the parcel data has been updated


### Update Special District Fields

In [17]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_SpecialDistrict):
    arcpy.Delete_management(ParcelPoint_SpecialDistrict)

print("Starting the Special Planning District Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_SpecialDistrict, ParcelPoint_SpecialDistrict, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Special Planning District Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'SPECIAL_PLANNING_DISTRICT'], ParcelPoint_SpecialDistrict, ['APN', 'NAME'])
print ("The 'SPECIAL_PLANNING_DISTRICT' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'SPECIAL_PLANNING_DISTRICT_TYPE'], ParcelPoint_SpecialDistrict, ['APN', 'TYPE'])
print ("The 'SPECIAL_PLANNING_DISTRICT_TYPE' field in the parcel data has been updated")

Starting the Special Planning District Spatial Join
Finished the Special Planning District Spatial Join
Started data transfer: 2019-04-25 13:40:31
Finished data transfer: 2019-04-25 13:45:15
The 'SPECIAL_PLANNING_DISTRICT' field in the parcel data has been updated
Started data transfer: 2019-04-25 13:45:15
Finished data transfer: 2019-04-25 13:49:59
The 'SPECIAL_PLANNING_DISTRICT_TYPE' field in the parcel data has been updated


### Update 1987 Index Fields

In [18]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Index1987):
    arcpy.Delete_management(ParcelPoint_Index1987)

print("Starting the 1987 Index Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Index1987, ParcelPoint_Index1987, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the 1987 Index Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'INDEX_1987'], ParcelPoint_Index1987, ['APN', 'MAP_NUMBER'])
print ("The 'INDEX_1987' field in the parcel data has been updated")
fieldJoinCalc(ParcelLayer, ['APN', 'INDEX_1987_HYPERLINK'], ParcelPoint_Index1987, ['APN', 'MAP_PATH'])
print ("The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated")

Starting the 1987 Index Spatial Join
Finished the 1987 Index Spatial Join
Started data transfer: 2019-04-25 13:50:27
Finished data transfer: 2019-04-25 13:55:02
The 'INDEX_1987' field in the parcel data has been updated
Started data transfer: 2019-04-25 13:55:02
Finished data transfer: 2019-04-25 13:59:31
The 'INDEX_1987_HYPERLINK' field in the parcel data has been updated


### Update Postal Town Field

In [19]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_PstlTown):
    arcpy.Delete_management(ParcelPoint_PstlTown)

print("Starting the Postal Town Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_UrbanArea, ParcelPoint_PstlTown, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Town Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PSTL_TOWN'], ParcelPoint_PstlTown, ['APN', 'Name'])
print ("The 'PSTL_TOWN' field in the parcel data has been updated")

Starting the Postal Town Spatial Join
Finished the Postal Town Spatial Join
Started data transfer: 2019-04-25 13:59:41
Finished data transfer: 2019-04-25 14:04:24
The 'PSTL_TOWN' field in the parcel data has been updated


### Update Postal Zip Field

In [20]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_PstlZip):
    arcpy.Delete_management(ParcelPoint_PstlZip)

print("Starting the Postal Zip Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Zip, ParcelPoint_PstlZip, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Postal Zip Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'PSTL_ZIP5'], ParcelPoint_PstlZip, ['APN', 'GEOID10'])
print ("The 'PSTL_ZIP5' field in the parcel data has been updated")

Starting the Postal Zip Spatial Join
Finished the Postal Zip Spatial Join
Started data transfer: 2019-04-25 14:04:34
Finished data transfer: 2019-04-25 14:09:06
The 'PSTL_ZIP5' field in the parcel data has been updated


### Add 'CSLT' to Jurisdiction field

#### Pseudo Code
* Select All Parcels that have their centroid within CSLT Boundary
* For selected features, change JURISDICTION value to 'CSLT'
* Make sure SDE Fields can take 4 characters

In [ ]:
csltParcels = arcpy.SelectLayerByLocation_management(ParcelLayer, "HAVE_THEIR_CENTER_IN", sde_CSLT)


### Update Within TRPA Boundary Field

In [21]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_TRPAboundary):
    arcpy.Delete_management(ParcelPoint_TRPAboundary)

print("Starting the TRPA Boundary Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_TRPAboundary, ParcelPoint_TRPAboundary, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the TRPA Boundary Spatial Join")

# transfer attributes to Parcel Layer
fieldJoinCalc(ParcelLayer, ['APN', 'WITHIN_TRPA_BNDY'], ParcelPoint_TRPAboundary, ['APN', 'WITHIN_TRPA_BNDY'])
print ("The 'WITHIN_TRPA_BNDY' field in the parcel data has been updated")

Starting the TRPA Boundary Spatial Join
Finished the TRPA Boundary Spatial Join
Started data transfer: 2019-04-25 14:09:16
Finished data transfer: 2019-04-25 14:14:34
The 'WITHIN_TRPA_BNDY' field in the parcel data has been updated


### Update Littoral Field

#### Pseudo Code
* Create a Littoral Feature class to use in selecting parcels
* Select by Location parcels that have their center within the new littoral feature
* Update Littoral Field with '1' for Littoral and '0' for non-Littoral

In [ ]:
# delete in memory output feature class if it exists already
if arcpy.Exists(ParcelPoint_Littoral):
    arcpy.Delete_management(ParcelPoint_Littoral)

print("Starting the Littoral Spatial Join")
# Process Fire District Spatial Join
arcpy.SpatialJoin_analysis(ParcelPoint, sde_Littoral, ParcelPoint_Littoral, "JOIN_ONE_TO_ONE", "KEEP_ALL", "", "HAVE_THEIR_CENTER_IN", "", "")
print ("Finished the Littoral Spatial Join")

with arcpy.da.UpdateCursor(ParcelLayer, "LITTORAL") as cursor:
    for row in cursor:
        if row

### Update Allowed Coverage Field (based on Bailey)

In [8]:
import arcpy
arcpy.env.overwriteOutput = True

# SDE file path 
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# SDE feature class
sde_Bailey = sdeBase + "\\sde.SDE.Soils\\sde.SDE.land_capability_Bailey_Soils"

# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# create out table for the stats sum
outTable =  "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Bailey_Table"

# Create Identity Output Layer
id_ParcelLyr_BaileyLyr = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Bailey"

# Create Impervious Layer
Bailey_lyr = wk_memory + "Bailey_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(Bailey_lyr):
    arcpy.Delete_management(Bailey_lyr)
    print ("Deleted Bailey Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_BaileyLyr):
    arcpy.Delete_management(id_ParcelLyr_BaileyLyr)
    print ("Deleted Parcel Bailey Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Bailey, Bailey_lyr)
print ("Created feature layer of Bailey Soils")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, Bailey_lyr, id_ParcelLyr_BaileyLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_BaileyLyr, identity_layer, where_clause = "NOT CAPABILITY in ('WB', '-1', '0')")
print ("Created identity feature layer of bailey")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Square Footage all polygons")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['CAPABILITY', 'SqFt', 'PERCENT_COVERAGE_ALLOWED']) as cur:
    for row in cur:
        if row[0] != ('','WB'):
            row[1] = row[1]*row[2]
        else:
            row[1] == 0
        cur.updateRow(row)
    print("Calculated Allowed Square Footage")
    
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN")
print ("Summed Square Footage of Bailey")

## Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
fieldJoinCalc(ParcelLayer, ['APN', 'ALLOWABLE_COVERAGE_BAILEY_SQFT'], outTable, ['APN', 'SUM_SqFt'])
print ("The 'ALLOWABLE_COVERAGE_BAILEY_SQFT' field in the parcel data has been updated")

Created feature layer of Bailey Soils
Starting Identity
Finished Identity
Created identity feature layer of bailey
Calculated Square Footage all polygons
Calculated Allowed Square Footage
Summed Square Footage of Bailey
Started data transfer: 2019-06-25 13:33:36
Finished data transfer: 2019-06-25 13:33:39
The 'ALLOWABLE_COVERAGE_BAILEY_SQFT' field in the parcel data has been updated


### Update Land Cabality Field

In [23]:
# Import system modules
import arcpy
import os

# Set local variables
workspace = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb"

arcpy.env.overwriteOutput = True

# Want to join Land Capability to Parcels
targetFeatures = os.path.join(workspace, "Parcels_Base")
joinFeatures = os.path.join(workspace, "NRCS2007")

# Output will be the target features, states, with a mean city population field (mcp)
outfc = os.path.join(workspace, "spjn_Parcel_NRCS")

# Create a new fieldmappings and add the two input feature classes.
fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(targetFeatures)
fieldmappings.addTable(joinFeatures)

# get the field mappings for 
landCapFieldIndex = fieldmappings.findFieldMapIndex("Land_Capab")
fieldmap = fieldmappings.getFieldMap(landCapFieldIndex)

# Get the output field's properties as a field object
field = fieldmap.outputField

# Rename the field and pass the updated field object back into the field map
field.name = "LandCapability"
field.aliasName = "Land Capability"
field.length = 500
fieldmap.outputField = field

# Set the merge rule to join with a comma delimiter and then replace the old fieldmap in the mappings object with the updated one
fieldmap.mergeRule = "join"
fieldmap.joinDelimiter = ','
fieldmappings.replaceFieldMap(landCapFieldIndex, fieldmap)

#Run the Spatial Join tool, using the defaults for the join operation and join type
arcpy.SpatialJoin_analysis(targetFeatures, joinFeatures, outfc, "#", "#", fieldmappings)
print ("Spatial join complete.")

spjnFC = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\spjn_Parcel_NRCS"
# remove duplicate land capabilities
with arcpy.da.UpdateCursor(spjnFC, ['CAPABILITY']) as cur:
    for row in cur:
        if row[0] is not None:
            v = row[0]
            v = v.split(',')
            v = list(set(v))
            v = ','.join(v)
            row[0] = v
            cur.updateRow(row)
    print ("Duplicate values removed.")

RuntimeError: FieldMappings: Error in adding table to field mappings

### Update Impervious Coverage Field

In [5]:
import arcpy
arcpy.env.overwriteOutput = True

# SDE file path 
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# SDE feature class
sde_Impervious = sdeBase + "\\sde.SDE.Tahoe_Impervious_2010\\sde.SDE.Impervious_2010"

# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# create out table for the stats sum
outTable =  "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\id_Parcel_Imp_Table"

# Create Identity Output Layer
id_ParcelLyr_ImperviousLyr = wk_memory + "id_Parcel_Impervious"

# Create Impervious Layer
Impervious_lyr = wk_memory + "Impervious_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(Impervious_lyr):
    arcpy.Delete_management(Impervious_lyr)
    print ("Deleted Impervious Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_ImperviousLyr):
    arcpy.Delete_management(id_ParcelLyr_ImperviousLyr)
    print ("Deleted Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_Impervious, Impervious_lyr, where_clause = "FType IN (4,5)")
print ("Created impervious feature layer of buildings and other impervious surface")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, Impervious_lyr, id_ParcelLyr_ImperviousLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_ImperviousLyr, identity_layer, where_clause = "FType IN (4,5)")
print ("Created identity feature layer of buildings and other impervious surface")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SqFt", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Impervious Square Footage")
                                                           
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["SqFt", "SUM"]], "APN")
print ("Summed Square Footage of Impervious")

# Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
fieldJoinCalc(ParcelLayer, ['APN', 'IMPERVIOUS_SURFACE_SQFT'], outTable, ['APN', 'SUM_SqFt'])
print ("The 'ImperviousCoverage_SqFt' field in the parcel data has been updated")

Deleted Impervious Layer
Deleted Identity Layer
Deleted Out Table
Created impervious feature layer of buildings and other impervious surface
Starting Identity
Finished Identity
Created identity feature layer of buildings and other impervious surface
Calculated Impervious Square Footage
Summed Square Footage of Impervious
Started data transfer: 2019-07-22 11:45:23
Finished data transfer: 2019-07-22 11:47:56
The 'ImperviousCoverage_SqFt' field in the parcel data has been updated


### Update Land Capability Coverage Allowed field (based on LCV layer)

#### Pseudo Code
* Push value to Parcel_Master and Parcel_Simplified

In [ ]:
import arcpy
arcpy.env.overwriteOutput = True
def fieldJoinCalc(updateFC, updateFieldsList, sourceFC, sourceFieldsList):
    from time import strftime  
    print ("Started data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))
    
    #updateFC = r"C:\Path\UpdateFeatureClass"  
    #updateFieldsList = ["JoinField", "ValueField"]
    
    # sourceFC = r"C:\Path\SourceFeatureClass"  
    #sourceFieldsList = ["JoinField", "ValueField"]  
    
    # Use list comprehension to build a dictionary from a da SearchCursor  
    valueDict = {r[0]:(r[1:]) for r in arcpy.da.SearchCursor(sourceFC, sourceFieldsList)}  
   
    with arcpy.da.UpdateCursor(updateFC, updateFieldsList) as updateRows:  
        for updateRow in updateRows:  
            # store the Join value of the row being updated in a keyValue variable  
            keyValue = updateRow[0]  
            # verify that the keyValue is in the Dictionary  
            if keyValue in valueDict:  
                # transfer the value stored under the keyValue from the dictionary to the updated field.  
                updateRow[1] = valueDict[keyValue][0]  
                updateRows.updateRow(updateRow)    
    del valueDict  
    print ("Finished data transfer: " + strftime("%Y-%m-%d %H:%M:%S"))

# SDE file path 
sdeBase = "F:\\GIS\\GIS_DATA\\Vector.sde"
sdeCollect ="F:\\GIS\\GIS_DATA\\Collection.sde"

# in memory output file path
wk_memory = "in_memory" + "\\"

# # LCV feature class
# sde_LCV = sdeCollect + "\\sde_collection.SDE.LandCapabilityWebApp\\sde_collection.SDE.Land_Capability_Verification"

# parcel feature classes
# parcel layer
ParcelLayer = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
# ParcelLayer = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
# ParcelSimple = sdeBase + "\\sde.SDE.Parcels\\sde.SDE.Parcels_Simplified"

# # create out table for the stats sum
# outTable =  "F:\\GIS\\PROJECTS\\CurrentPlanning\\LCV_WebMap\\Data\\ProjectData.gdb\\id_Parcel_LCV_Table"

# # Create Identity Output Layer
# id_ParcelLyr_LCVLyr = "F:\\GIS\\PROJECTS\\CurrentPlanning\\LCV_WebMap\\Data\\ProjectData.gdb\\id_Parcel_LCV"

# Create Impervious Layer
LCV_lyr = wk_memory + "LCV_lyr"

# Create Identity Layer
identity_layer = wk_memory + "identity_layer"

# delete in memory output feature class if it exists already
if arcpy.Exists(LCV_lyr):
    arcpy.Delete_management(LCV_lyr)
    print ("Deleted in memory LCV Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(id_ParcelLyr_LCVLyr):
    arcpy.Delete_management(id_ParcelLyr_LCVLyr)
    print ("Deleted Parcel LCV Identity Layer")

# delete in memory output feature class if it exists already
if arcpy.Exists(identity_layer):
    arcpy.Delete_management(identity_layer)
    print ("Deleted identity_layer Layer")
    
# delete in memory output feature class if it exists already
if arcpy.Exists(outTable):
    arcpy.Delete_management(outTable)
    print ("Deleted Out Table")
    
# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(sde_LCV, LCV_lyr)
print ("Created feature layer of Bailey LCV Layer")

# Process: Use the Identity function
print ("Starting Identity")
arcpy.Identity_analysis (ParcelLayer, LCV_lyr, id_ParcelLyr_LCVLyr)
print ("Finished Identity")

# Make a layer from the feature class Impervious that only passes Ftype = 'building' and 'other'
arcpy.MakeFeatureLayer_management(id_ParcelLyr_LCVLyr, identity_layer, where_clause = "NOT LCV_IPES in ('WB', '-1', '0','IPES')")
print ("Created identity feature layer of parcels and lcv identity output")

# calculate geometry of output identity
arcpy.CalculateField_management(identity_layer, "SQFT", "!shape.area@SQUAREFEET!", "PYTHON3", "")
print ("Calculated Square Footage of all polygons")

# multiply square footage by bailey coefficents
with arcpy.da.UpdateCursor(identity_layer, ['LCV_IPES', 'SQFT', 'Allowed_Coverage']) as cur:
    for row in cur:
        if row[0] != ('','WB', 'IPES'):
            if row[0] in ('1A','1B','1C','2'):
                row[2] = row[1] * 0.01
            elif row[0] == '3':
                row[2] = row[1] * 0.05
            elif row[0] == '4':
                row[2] = row[1] * 0.2
            elif row[0] == '5':
                row[2] = row[1] * 0.25
            elif row[0] in ('6','7'):
                row[2] = row[1] * 0.3
        else:
            row[2] == 0
        cur.updateRow(row)
    print("Calculated Coverage Allowed (sq.ft.)")
    
# Sum the square footage of buildings and other by APN
arcpy.Statistics_analysis(identity_layer, outTable, [["Allowed_Coverage", "SUM"]], "APN")
print ("Summed Square Footage of Coverage Allowed")

edit = arcpy.da.Editor(sdeBase)
print ("edit created")
try:
    edit.startEditing()
    print("edit started")
    edit.startOperation()
    print("operation started")
    # Perform edits
    ## Join parcel sums back to parcel layer and calculate field "Impervious Surface Sq Ft"
    fieldJoinCalc(ParcelSimple, ['APN', 'COVERAGE_ALLOWED'], outTable, ['APN', 'SUM_Allowed_Coverage'])
    print("The 'COVERAGE_ALLOWED' field in the parcel simplified data has been updated")
    edit.stopOperation()
    print("operation stopped")
    edit.stopEditing(True)  ## Stop the edit session with True to save the changes
    print("edit stopped")
except Exception as err:
    print(err)
    if edit.isEditing:
        edit.stopOperation()
        print("operation stopped in except")
        edit.stopEditing(False)  ## Stop the edit session with False to abandon the changes
        print("edit stopped in except")
finally:
    # Cleanup
    arcpy.ClearWorkspaceCache_management()

### Update Land Use fields

In [25]:
import arcpy

fc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
fields = ("COUNTY_LANDUSE_CODE", "COUNTY_LANDUSE_DESCRIPTION", "TRPA_LANDUSE_DESCRIPTION", 'JURISDICTION')

with arcpy.da.UpdateCursor(fc, fields) as cursor:
    for row in cursor:
        ctyluc = row[0]
        cty = row[3]
# set Washoe county land use code
        # set TRPA Land Use Description
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('400', '410', '440', '500', '510', '520', '630', '640', '670', '720'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '250'):
                row[2] = "Condominium"
            elif ctyluc in ('240'):
                row[2] = "Condominium Common Area"
            elif ctyluc in ('220', '230', '300', '310', '320', '330', '340', '350', '360'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('600', '620'):
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', 'PBRD'):
                row[2] = "Public Service"
            elif ctyluc in ('190'):
                row[2] = "Recreation"
            elif ctyluc in ('200'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '120', '130', '140', '150', '160', '170', '180'):
                row[2] = "Vacant"            
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        # set county land use description field
        if (row[0] != None or row[0] != "") and (row[3] == 'WA'):
            if ctyluc in ('710'):
                row[1] = "Intracounty public utility"
            elif ctyluc == '700':
                row[1] = 'Centrally assessed public utility'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial: retail or office with Indus'
            elif ctyluc == '500':
                row[1] = 'General industrial: light indust, trucking, warehs'
            elif ctyluc == '440':
                row[1] = 'Resort commercial: ski, golf, sports, etc.'
            elif ctyluc == '430':
                row[1] = 'Commercial hotel or motel'
            elif ctyluc == '420':
                row[1] = 'Casino or hotel casino'
            elif ctyluc == '410':
                row[1] = 'Offices, professional and business, banks, etc.'
            elif ctyluc == '400':
                row[1] = 'General Commercial: retail, mixed, parking, school'
            elif ctyluc == '340':
                row[1] = 'Ten or more units'
            elif ctyluc == '330':
                row[1] = 'Five to Nine Units'
            elif ctyluc == '320':
                row[1] = 'Three or four Units'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '250':
                row[1] = 'Condo or Townhouse valued as apartment use'
            elif ctyluc == '240':
                row[1] = 'Common Area'
            elif ctyluc == '210':
                row[1] = 'Condominium or Townhouse'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Public Parks: vacant or improved'
            elif ctyluc == '170':
                row[1] = 'Other, unbuildable: roads, restrictions, terrain'
            elif ctyluc == '160':
                row[1] = 'Splinter, unbuildable: small size or shape'
            elif ctyluc == '140':
                row[1] = 'Vacant, commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant, multi-residential'
            elif ctyluc == '120':
                row[1] = 'Vacant, single family'
            elif ctyluc == '110':
                row[1] = 'Vacant, under development'
            elif ctyluc == '100':
                row[1] = 'Vacant, other or unknown'            
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# Set Carson City County Land Use Descriptions
        # set TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc in ('400', '401', '402', '403', '404', '408', '410', '411', 
                          '412', '440', '441', '460', '470', '480', '482', '490', 
                          '500', '501', '510', '511', '512', '513', '520', '521', 
                          '560', '570', '580', '582', '590', '624', '625', '694', 
                          '800', '820', '830', '840', '880', '882', '890', '920', 
                          '921', '930', '960', '980', '990'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                trpalucdesc = "Condominium"
            elif ctyluc == '970':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('240', '241', '300', '301', '310', '311', '313', '320', 
                            '321', '330', '331', '333', '340', '341', '350', '360', 
                            '370', '380', '382', '390', '698'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('190', '600', '610', '612', '613', '614', '615', '616', 
                            '618', '620', '695', '696', '697', '810'):
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', '711', '720', '731', '732', '733', '780', 
                            '790', '910', '922'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '201', '220', '222', '230', '231', '232', '260', 
                            '270', '280', '282', '290', '622', '692', '693'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '421', '430', '431', '432', '514'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '108', '110', '117', '120', '130', '140', '150', '160'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        # set county LUC Description
        if (row[0] != None or row[0] != "") and (row[3] == 'CC'):
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential' 
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# update Douglas Land Use descriptions        
        #Update TRPA Land Use description
        if (row[0] != None or row[0] != "") and (row[3] == 'DG'):
            if ctyluc in ('400', '402', '410', '411', '412', 
                          '440', '460', '470', '480', '500', 
                          '510', '560', '580', '582'):
                row[2] = "Commercial"
            elif ctyluc in ('210', '211'):
                row[2] = "Condominium"
            elif ctyluc == '270':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('300', '310', '320', '330', '350', '390'):
                row[2] = "Multi-Family Residential"
            elif ctyluc == '190':
                row[2] = "Open Space"
            elif ctyluc in ('700', '710', '711', '910', '980', '970'):
                row[2] = "Public Service"
            elif ctyluc in ('450', '900', '970'):
                row[2] = "Recreation"
            elif ctyluc in ('200', '220', '230', '236', '240', '280', '282'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('420', '430'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('100', '110', '117', '120', '130', '140'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
        if (row[0] != None or row[0] != "" or ctyluc.isspace() != True) and (row[3] == 'DG'):
            # set County land use description
            if ctyluc == '980':
                row[1] = 'Special Purpose with Minor Improvements'
            elif ctyluc == '970':
                row[1] = 'Special Purpose Common Area'
            elif ctyluc == '910':
                row[1] = 'Cemeteries'
            elif ctyluc == '900':
                row[1] = 'Parks for Public Use'
            elif ctyluc == '711':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature Under Construction'
            elif ctyluc == '710':
                row[1] = 'Communication, Transportation, and Utility Property of a Local Nature'
            elif ctyluc == '700':
                row[1] = 'Operating Communication, Transportation, and Utility Property of an Interstate or Intercounty Nature'
            elif ctyluc == '582':
                row[1] = 'Industrial with Minor Improvements - with structures insufficient to determine intended use'
            elif ctyluc == '580':
                row[1] = 'Industrial with Minor Improvements'
            elif ctyluc == '560':
                row[1] = 'Industrial Auxiliary Area'
            elif ctyluc == '510':
                row[1] = 'Commercial Industrial - retail or office use combined with Industrial use'
            elif ctyluc == '500':
                row[1] = 'General Industrial - light industry, trucking and warehousing, service, repair, etc.'
            elif ctyluc == '480':
                row[1] = 'Commercial with Minor Improvements'
            elif ctyluc == '470':
                row[1] = 'Commercial Common Area'
            elif ctyluc == '460':
                row[1] = 'Commercial Auxiliary Area'
            elif ctyluc == '450':
                row[1] = 'Golf Course'
            elif ctyluc == '440':
                row[1] = 'Commercial Recreation'
            elif ctyluc == '430':
                row[1] = 'Commercial Living Accommodations'
            elif ctyluc == '420':
                row[1] = 'Casino or Hotel Casino'
            elif ctyluc == '410':
                row[1] = 'Offices, Professional and Business Services'
            elif ctyluc == '402':
                row[1] = 'Parking and/or Parking Structures'
            elif ctyluc == '400':
                row[1] = 'General Commercial'
            elif ctyluc == '390':
                row[1] = 'Mixed Use with Multi-Family Residential as primary use'
            elif ctyluc == '350':
                row[1] = 'Manufactured Home Park - Ten or More Manufactured Home Units'
            elif ctyluc == '330':
                row[1] = 'Five or More Units - Low Rise'
            elif ctyluc == '320':
                row[1] = 'Three to Four Units'
            elif ctyluc == '310':
                row[1] = 'Two Single Family Units'
            elif ctyluc == '300':
                row[1] = 'Duplex'
            elif ctyluc == '282':
                row[1] = 'Single Family Residential with Minor Improvements - No livable structures'
            elif ctyluc == '280':
                row[1] = 'Single Family Residential with Minor Improvements'
            elif ctyluc == '270':
                row[1] = 'Single Family Residential Common Area'
            elif ctyluc == '240':
                row[1] = 'Individual Residential Unit - Townhouse or Row House'
            elif ctyluc == '236':
                row[1] = 'Personal Property Manufactured Home Secured'
            elif ctyluc == '230':
                row[1] = 'Personal Property Manufactured Home on the Unsecured Roll'
            elif ctyluc == '220':
                row[1] = 'Manufactured Home Converted to Real Property'
            elif ctyluc == '210':
                row[1] = 'Individual Unit in a Multiple Unit Building'
            elif ctyluc == '200':
                row[1] = 'Single Family Residence'
            elif ctyluc == '190':
                row[1] = 'Vacant - Public Use Lands'
            elif ctyluc == '140':
                row[1] = 'Vacant - Commercial'
            elif ctyluc == '130':
                row[1] = 'Vacant - Multi-Residential'
            elif ctyluc == '120':
                row[1] = 'Vacant - Single Family Residential'
            elif ctyluc == '117':
                row[1] = 'Vacant - Roads/Easements'
            elif ctyluc == '110':
                row[1] = 'Vacant - Splinter and Other Unbuildable'
            elif ctyluc == '100':
                row[1] = 'Vacant - Unknown/Other'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
# Set El Doroda County Land Use Descrition fields
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc in ('03', '29', '31', '32', '34', '36', '37', '38', '39', '41', '42', '43', '44', '45', '46', '47', '48', 
                          '65', '67', '68', '82', '91', '93'):
                row[2] = "Commercial"
            elif ctyluc == '14':
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('01', '07', '12', '13', '16', '18', '19', '28', '35'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('25', '26', '50', '51', '52', '55', '56', '60', '70', '75', '79'):
                row[2] = "Open Space"
            elif ctyluc in ('90', '92', '94', '96', '97', '98'):
                row[2] = "Public Service"
            elif ctyluc in ('61', '62', '63', '64'):
                row[2] = "Recreation"
            elif ctyluc in ('06', '11', '15', '22', '23'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('33', '80', '81'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '02', '05', '17', '21', '24', '30', '40'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
# set county land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'EL'):
            if ctyluc == '98':
                row[1] = 'DEV MSC FIRE SUPPRESSION FACILITIES'
            elif ctyluc == '96':
                row[1] = 'DEV MSC CEMETERIES'
            elif ctyluc == '94':
                row[1] = 'DEV MSC SCHOOLS - LARGE (101+ STUDENTS)'
            elif ctyluc == '93':
                row[1] = 'DEV MSC SCHOOLS - MEDIUM (13-100 STUDENTS)'
            elif ctyluc == '92':
                row[1] = 'DEV MSC SCHOOLS - SMALL (1-12 STUDENTS)'
            elif ctyluc == '90':
                row[1] = 'UTL IND PUBLIC UTILITY (ON STATE ASSESSED ROLL)'
            elif ctyluc == '84':
                row[1] = 'DEV MSC TEMPORARY USE CODE FOR PROJECT 184'
            elif ctyluc == '82':
                row[1] = 'DEV COM PARKING LOT'
            elif ctyluc == '81':
                row[1] = 'DEV MSC UNDERLYING INTEREST IN TIME SHARE PROJ'
            elif ctyluc == '79':
                row[1] = 'RLU MSC ENV. SENSITIVE LAND - RESTRICTED USE'
            elif ctyluc == '68':
                row[1] = 'DEV COM MARINAS'
            elif ctyluc == '65':
                row[1] = 'DEV COM RESTAURANT'
            elif ctyluc == '64':
                row[1] = 'DEV MSC SKI RESORTS'
            elif ctyluc == '63':
                row[1] = 'DEV MSC CAMPGROUNDS'
            elif ctyluc == '62':
                row[1] = 'DEV MSC COMMUNITY ORIENTED FACILITIES'
            elif ctyluc == '61':
                row[1] = 'DEV MSC MISC. IMPROVED RECREATIONAL'
            elif ctyluc == '60':
                row[1] = 'VAC MSC VACANT RECREATIONAL LAND'
            elif ctyluc == '50':
                row[1] = 'TPZ MSC TIMBER PRESERVE ZONING - ACTIVE'
            elif ctyluc == '48':
                row[1] = 'DEV IND OFFICES'
            elif ctyluc == '47':
                row[1] = 'DEV IND HOSPITALS & CONVALESCENT HOSPITALS'
            elif ctyluc == '46':
                row[1] = 'DEV IND MEDICAL/DENTAL/VET OFFICES'
            elif ctyluc == '45':
                row[1] = 'DEV IND LIGHT MANUFACTURING'
            elif ctyluc == '43':
                row[1] = 'DEV IND WAREHOUSES'
            elif ctyluc == '42':
                row[1] = 'DEV IND MINI-WAREHOUSES (MINI-STORAGE)'
            elif ctyluc == '41':
                row[1] = 'DEV IND MISC. IMPROVED INDUSTRIAL PROPERTY'
            elif ctyluc == '40':
                row[1] = 'VAC IND VACANT INDUSTRIAL LAND'
            elif ctyluc == '39':
                row[1] = 'DEV COM SUPERMARKETS'
            elif ctyluc == '38':
                row[1] = 'DEV COM RETAIL STORES >15,000 SQ. FT.'
            elif ctyluc == '37':
                row[1] = 'DEV COM RETAIL STORES 5,001-15,000 SQ. FT.'
            elif ctyluc == '36':
                row[1] = 'DEV COM RETAIL STORES <=5,000 SQ. FT.'
            elif ctyluc == '35':
                row[1] = 'DEV COM MOBILE HOME PARKS'
            elif ctyluc == '34':
                row[1] = 'DEV COM SERVICE STATION'
            elif ctyluc == '33':
                row[1] = 'DEV COM MOTEL, HOTEL'
            elif ctyluc == '31':
                row[1] = 'DEV COM MISC. IMPROVED COMMERCIAL'
            elif ctyluc == '30':
                row[1] = 'VAC COM VACANT COMMERCIAL LAND'
            elif ctyluc == '29':
                row[1] = 'DEV MSC RURAL NON-RES. IMPROVEMENT 2.51-20.0 AC.'
            elif ctyluc == '26':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - NON-RENEWAL'
            elif ctyluc == '25':
                row[1] = 'AGP MSC RURAL RESTRICTIVE ZONING - CLCA (ACTIVE)'
            elif ctyluc == '24':
                row[1] = 'VAC RES RURAL RES. LAND 20+ MINOR NON-RES IMPR'
            elif ctyluc == '23':
                row[1] = 'DEV RES RURAL RES. 20+ AC. 1 RES. UNIT'
            elif ctyluc == '22':
                row[1] = 'DEV RES RURAL RES. 2.51-20.0 AC. 1 SF UNIT'
            elif ctyluc == '21':
                row[1] = 'VAC RES VAC RURAL RES LAND 2.51-20.0 AC. 1 UNIT'
            elif ctyluc == '17':
                row[1] = 'VAC MSC SUBJ. TO OPEN SPACE CONTRACT (NOT CLCA)'
            elif ctyluc == '16':
                row[1] = 'DEV RES MOBILE HOME ON RENTED LAND'
            elif ctyluc == '15':
                row[1] = 'DEV RES RESIDENCE ON LEASED LAND'
            elif ctyluc == '14':
                row[1] = 'DEV MFR CONDOMINIUMS & TOWNHOUSES'
            elif ctyluc == '13':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 4+ UNITS'
            elif ctyluc == '12':
                row[1] = 'DEV MFR MULTI-RESIDENTIAL 2-3 UNITS'
            elif ctyluc == '11':
                row[1] = 'DEV RES SINGLE FAM. RES. <=2.5 AC.(INC. MAN. HMS'
            elif ctyluc == '07':
                row[1] = 'DEV MFR RETIREMENT HOUSING'
            elif ctyluc == '05':
                row[1] = 'VAC MFR VACANT MULTI-RES. LAND 4+ UNITS ALLOWED'
            elif ctyluc == '03':
                row[1] = 'DEV COM PLACE OF WORSHIP'
            elif ctyluc == '02':
                row[1] = 'VAC RES NON-RES. IMPROVEMENTS <=2.5 AC.'
            elif ctyluc == '00':
                row[1] = 'VAC RES VACANT RES. LAND <=2.5 AC. 1-3 UNITS'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)

# set TRPA land use description
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            if ctyluc in ('07', '11', '12', '13', '14', '15', '17', '21', '22', '23', 
                          '24', '25', '26', '27', '29', '31', '32', '36', '37', '38', 
                          '39', '62', '63', '71', '88'):
                row[2] = "Commercial"
            elif ctyluc == '04':
                row[2] = "Condominium"
            elif ctyluc == '89':
                row[2] = "Condominium Common Area"
            elif ctyluc in ('02', '03', '04', '05', '09', '28'):
                row[2] = "Multi-Family Residential"
            elif ctyluc in ('56', '60', '87', '90'):
                row[2] = "Open Space"
            elif ctyluc in ('72', '76', '77', '81'):
                row[2] = "Public Service"
            elif ctyluc in ('65', '66', '67', '68', '69'):
                row[2] = "Recreation"
            elif ctyluc in ('01', '08', '16'):
                row[2] = "Single Family Residential"     
            elif ctyluc in ('06', '18', '64'):
                row[2] = "Tourist Accommodation"
            elif ctyluc in ('00', '10', '20', '30'):
                row[2] = "Vacant"
            elif ctyluc is None:
                row[2] == ''
        cursor.updateRow(row)
# set Placer county land use description 
        if (row[0] != None or row[0] != "") and (row[3] == 'PL'):
            # set county land use description
            if ctyluc == '90':
                row[1] = 'GREENBELT'
            elif ctyluc == '89':
                row[1] = 'COMMON AREA'
            elif ctyluc == '88':
                row[1] = 'HIGHWAYS, ROADS, STREETS'
            elif ctyluc == '87':
                row[1] = 'RIVERS, LAKES, RESERVOIR, CANAL'
            elif ctyluc == '81':
                row[1] = 'UTILITIES, PUBLIC & PRIVATE'
            elif ctyluc == '77':
                row[1] = 'CEMETERIES'
            elif ctyluc == '76':
                row[1] = 'MISC. PUBLIC BUILDINGS'
            elif ctyluc == '72':
                row[1] = 'SCHOOLS'
            elif ctyluc == '71':
                row[1] = 'CHURCHES'
            elif ctyluc == '69':
                row[1] = 'MISCELLANEOUS RECREATIONAL'
            elif ctyluc == '68':
                row[1] = 'CAMPS & PARKS, GENERAL'
            elif ctyluc == '67':
                row[1] = 'SKI FACILITY'
            elif ctyluc == '66':
                row[1] = 'GOLF COURSE'
            elif ctyluc == '65':
                row[1] = 'TENNIS, SWIMMING CLUBS'
            elif ctyluc == '64':
                row[1] = 'LODGES, HALLS'
            elif ctyluc == '63':
                row[1] = 'MARINA, PIER'
            elif ctyluc == '62':
                row[1] = 'THEATER, BOWLING ALLEY'
            elif ctyluc == '61':
                row[1] = 'NON-PROFIT CAMPS/PARKS'
            elif ctyluc == '60':
                row[1] = 'CONSERVATION EASEMENT RESTRICTIONS'
            elif ctyluc == '56':
                row[1] = 'TIMBERLAND, ZONED TPZ'
            elif ctyluc == '55':
                row[1] = 'TIMBERLAND, UNRESTRICTED'
            elif ctyluc == '39':
                row[1] = 'MISCELLANEOUS INDUSTRIAL'
            elif ctyluc == '38':
                row[1] = 'WAREHOUSE'
            elif ctyluc == '37':
                row[1] = 'MINI-STORAGE, COVERED STORAGE'
            elif ctyluc == '36':
                row[1] = 'UNCOVERED STORAGE, WRECKING YARD'
            elif ctyluc == '32':
                row[1] = 'HEAVY INDUSTRIAL'
            elif ctyluc == '31':
                row[1] = 'LIGHT INDUSTRIAL'
            elif ctyluc == '30':
                row[1] = 'VACANT INDUSTRIAL'
            elif ctyluc == '29':
                row[1] = "MISCELLANEOUS COMM'L"
            elif ctyluc == '28':
                row[1] = 'MOBILE HOME PARK'
            elif ctyluc == '27':
                row[1] = 'PARKING LOTS'
            elif ctyluc == '26':
                row[1] = 'AUTO SALES, REPAIR'
            elif ctyluc == '25':
                row[1] = 'SERVICE STATION'
            elif ctyluc == '24':
                row[1] = 'MINI-MARKET WITH GAS'
            elif ctyluc == '23':
                row[1] = "BANKS, S&L'S, CREDIT UNION"
            elif ctyluc == '22':
                row[1] = 'FAST FOOD RESTAURANT'
            elif ctyluc == '21':
                row[1] = 'RESTAURANTS, COCKTAIL LOUNGES'
            elif ctyluc == '20':
                row[1] = 'VACANT, COMMERCIAL'
            elif ctyluc == '19':
                row[1] = 'OFFICE MEDICAL/DENTAL'
            elif ctyluc == '18':
                row[1] = 'HOTELS, MOTELS, RESORTS'
            elif ctyluc == '17':
                row[1] = 'OFFICE GENERAL'
            elif ctyluc == '16':
                row[1] = 'RESIDENCE ON COMMERCIAL LAND'
            elif ctyluc == '15':
                row[1] = 'SHOPPING CENTER'
            elif ctyluc == '14':
                row[1] = 'OFFICE CONDO'
            elif ctyluc == '13':
                row[1] = 'MINI-MARKETS, NO GAS'
            elif ctyluc == '12':
                row[1] = 'SUBURBAN STORE'
            elif ctyluc == '11':
                row[1] = 'COMMERCIAL STORE'
            elif ctyluc == '10':
                row[1] = 'VACANT, SUBDIVIDED RESIDENTIAL'
            elif ctyluc == '09':
                row[1] = 'MOBILE HOME IN M H PARK'
            elif ctyluc == '08':
                row[1] = 'MOBILE HOME OUTSIDE OF PARK'
            elif ctyluc == '07':
                row[1] = 'RESIDENTIAL, AUXILIARY IMP'
            elif ctyluc == '06':
                row[1] = 'TIMESHARES'
            elif ctyluc == '05':
                row[1] = 'APARTMENTS, 4 UNITS OR MORE'
            elif ctyluc == '04':
                row[1] = 'SINGLE FAM RES, CONDO'
            elif ctyluc == '03':
                row[1] = '3 SINGLE FAM RES, TRIPLEX'
            elif ctyluc == '02':
                row[1] = '2 SINGLE FAM RES, DUPLEX'
            elif ctyluc == '01':
                row[1] = 'SINGLE FAM RES, HALF PLEX'
            elif ctyluc == '00':
                row[1] = 'VACANT, ALL TYPES-NOT ASGND'
            elif ctyluc is None:
                row[1] == ''
        cursor.updateRow(row)
    print ("Updated Land Use Description fields.")

Updated Land Use Description fields.


## Update SDE

### Update Parcel Point

In [4]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Point"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.ParcelPoints"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@XY']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
print("\nDisconnecting all users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))


Imported arcpy, os, csv, and datatime modules.

ArcInfo

Disconnecting all users...

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.ParcelPoints

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into SDE feature class
Inserting record 18000 into SDE feature class
Inserti

### Update Parcel Base

In [5]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcels_Base"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

fieldnames = ['APN', 'PPNO', 'JURISDICTION', 'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# disconnect all users
print("Disconnecting Users...")
arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")

# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# disconnect all users
arcpy.DisconnectUser(sdeBase, "ALL")

# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, os, csv, and datatime modules.

ArcInfo
Disconnecting Users...

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcels_Base

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into SDE feature class
Inserting record 18000 into SDE feature class
Inserting re

### Update Parcel Master

In [7]:
# import modules
import arcpy, os, csv
from datetime import datetime
# start timer
startTimer = datetime.now()
print("Imported arcpy, os, csv, and datatime modules.\n")
print(arcpy.ProductInfo())
## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

fieldnames =  ['APN', 'PPNO', 'HSE_NUMBR', 'UNIT_NUMBR', 'STR_DIR', 'STR_NAME', 
                'STR_SUFFIX', 'APO_ADDRESS', 'PSTL_TOWN', 'PSTL_STATE', 'PSTL_ZIP5', 
                'OWN_FIRST', 'OWN_LAST', 'OWN_FULL', 'MAIL_ADD1', 'MAIL_ADD2', 
                'MAIL_CITY', 'MAIL_STATE', 'MAIL_ZIP5', 'JURISDICTION','OWNERSHIP_TYPE', 
                'COUNTY_LANDUSE_CODE', 'COUNTY_LANDUSE_DESCRIPTION', 'TRPA_LANDUSE_DESCRIPTION', 
                'REGIONAL_LANDUSE', 'SOIL_1974', 'SOIL_2003', 'HRA_NAME', 'WATERSHED_NUMBER', 'WATERSHED_NAME', 'PRIORITY_WATERSHED',  
                'FIREPD', 'WITHIN_TRPA_BNDY', 'LITTORAL', 'AS_LANDVALUE', 'AS_IMPROVALUE', 
                'AS_SUM', 'TAX_LANDVALUE', 'TAX_IMPROVALUE', 'TAX_SUM', 'TAX_YEAR', 'PAS_ID', 
                'PAS_NAME', 'INDEX_1987', 'INDEX_1987_HYPERLINK', 'ALLOWABLE_COVERAGE_BAILEY_SQFT', 'IMPERVIOUS_SURFACE_SQFT', 'LOCAL_PLAN_HYPERLINK', 'DesignGuidelines_URL', 'ZONING', 
                'ZONING_DESCRIPTION', 'SINGLE_FAMILY_DENSITY', 'MULTI_FAMILY_DENSITY', 
                'TOURIST_ACCOMMODATION_DENSITY', 'BED_BREAKFAST_DENSITY', 'TIME_SHARE_DENSITY', 
                'COMMERCIAL_FLOOR_AREA_ALLOWED', 'SECONDARY_DWELLING_UNIT_ALLOWED', 'OVERLAY', 
                'MAX_CP_RES_DENSITY', 'MAX_CP_TAU_DENSITY', 'SPECIAL_PLAN_AREA_OVERLAY', 'TOWN_CENTER', 
                'SPECIAL_AREA', 'SPECIAL_PLANNING_DISTRICT', 'SPECIAL_PLANNING_DISTRICT_TYPE', 
                'LOCATION_TO_TOWNCENTER', 'PARCEL_ACRES', 'PARCEL_SQFT', 'SHAPE@']
#--------------------------------------------------------------------------------------------------------#

# # disconnect all users
# print("\nDisconnecting all users...")
# arcpy.DisconnectUser(sdeBase, "ALL")

print ("Unregistering feature dataset as versioned...")
# unregister the sde feature class as versioned
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")
print ("Finished unregistering feature dataset as versioned.")
# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(inputfc, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# # disconnect all users
# print("\nDisconnecting all users...")
# arcpy.DisconnectUser(sdeBase, "ALL")

print("\nRegistering feature dataset as versioned...")
# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")
print("\nFinished registering feature dataset as versioned.")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, os, csv, and datatime modules.

ArcInfo
Unregistering feature dataset as versioned...
Finished unregistering feature dataset as versioned.

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcel_Master

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16000 into SDE feature class
Inserting record 17000 into

### Update Parcel_Simplified

In [3]:
# Import system modules
import arcpy
import arcpy.management as DM
import arcpy.cartography as CA
import os, csv
from datetime import datetime

# start timer
startTimer = datetime.now()
print("Imported arcpy, arcpy.management as DM, arcpy.cartography as CA, os, csv, and datatime modules.\n")

# license used
print(arcpy.ProductInfo())

## Set the Local Variables
#------------------------------------------------------------------------------------------------------#
# set overwrite files envrionment setting to True
arcpy.env.overwriteOutput = True

# in memory location
wk_memory = "in_memory" + "\\"

# feature dataset to unversion and register as version
fdata = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels"

# Change this to the path of your input feature class
inputfc = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"

# in memory simplefied parcel master
simplifiedFeatures = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Simple"

# Change this to the path of your output FC
outfc = "F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcels_Simplified"
 
# set field map
fieldnames =  ['APN','PPNO','JURISDICTION','APO_ADDRESS','OWN_FULL','OWNERSHIP_TYPE','TRPA_LANDUSE_DESCRIPTION',
               'TOWN_CENTER', 'INDEX_1987_HYPERLINK','LOCAL_PLAN_HYPERLINK', 'DesignGuidelines_URL',
               'ZONING', 'ZONING_DESCRIPTION', 'SPECIAL_PLAN_AREA_OVERLAY', 'SPECIAL_AREA',
               'PARCEL_SQFT', 'SHAPE@']

#--------------------------------------------------------------------------------------------------------# 
# create in memory simple parcels
CA.SimplifyPolygon(inputfc, simplifiedFeatures, "POINT_REMOVE", 1, "#", "#", "KEEP_COLLAPSED_POINTS")

# # disconnect all users
# print("\nDisconnecting all users...")
# arcpy.DisconnectUser(sdeBase, "ALL")

# unregister the sde feature class as versioned
print ("Unregistering feature dataset as versioned...")
arcpy.UnregisterAsVersioned_management(fdata,"NO_KEEP_EDIT","COMPRESS_DEFAULT")
print ("Finished unregistering feature dataset as versioned.")

# deletes all rows from the SDE feature class
arcpy.TruncateTable_management(outfc)
print ("\nDeleted all records in: {}\n".format(outfc))

# insert rows from Temporary feature class to SDE feature class
with arcpy.da.InsertCursor(outfc, fieldnames) as oCursor:
    count = 0
    with arcpy.da.SearchCursor(simplifiedFeatures, fieldnames) as iCursor:
        for row in iCursor:
            oCursor.insertRow(row)
            count += 1
            if count % 1000 == 0:
                print("Inserting record {0} into SDE feature class".format(count))

# # disconnect all users
# print("\nDisconnecting all users...")
# arcpy.DisconnectUser(sdeBase, "ALL")

print("Registering feature dataset as versioned...")
# register SDE feature class as versioned
arcpy.RegisterAsVersioned_management(fdata, "NO_EDITS_TO_BASE")
print("Finished registering feature dataset as versioned.")

# confirm feature class was created
print("\nUpdated " + outfc)

# report how long it took to run the script
endTimer = datetime.now() - startTimer
print ("\nTime it took to run this script: {}".format(endTimer))

Imported arcpy, arcpy.management as DM, arcpy.cartography as CA, os, csv, and datatime modules.

ArcInfo
Unregistering feature dataset as versioned...
Finished unregistering feature dataset as versioned.

Deleted all records in: F:\GIS\GIS_DATA\Vector.sde\sde.SDE.Parcels\sde.SDE.Parcels_Simplified

Inserting record 1000 into SDE feature class
Inserting record 2000 into SDE feature class
Inserting record 3000 into SDE feature class
Inserting record 4000 into SDE feature class
Inserting record 5000 into SDE feature class
Inserting record 6000 into SDE feature class
Inserting record 7000 into SDE feature class
Inserting record 8000 into SDE feature class
Inserting record 9000 into SDE feature class
Inserting record 10000 into SDE feature class
Inserting record 11000 into SDE feature class
Inserting record 12000 into SDE feature class
Inserting record 13000 into SDE feature class
Inserting record 14000 into SDE feature class
Inserting record 15000 into SDE feature class
Inserting record 16

In [4]:
import arcpy
table ="F:\\GIS\\GIS_DATA\\Vector.sde\\sde.SDE.Parcels\\sde.SDE.Parcel_Master"
field_names = [field.name for field in arcpy.ListFields(simplifiedFeatures)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

0 OBJECTID
1 Shape
2 APN
3 PPNO
4 HSE_NUMBR
5 UNIT_NUMBR
6 STR_DIR
7 STR_NAME
8 STR_SUFFIX
9 APO_ADDRESS
10 PSTL_TOWN
11 PSTL_STATE
12 PSTL_ZIP5
13 OWN_FIRST
14 OWN_LAST
15 OWN_FULL
16 MAIL_ADD1
17 MAIL_ADD2
18 MAIL_CITY
19 MAIL_STATE
20 MAIL_ZIP5
21 JURISDICTION
22 COUNTY
23 OWNERSHIP_TYPE
24 COUNTY_LANDUSE_CODE
25 COUNTY_LANDUSE_DESCRIPTION
26 TRPA_LANDUSE_DESCRIPTION
27 REGIONAL_LANDUSE
28 SOIL_1974
29 SOIL_2003
30 ALLOWABLE_COVERAGE_BAILEY_SQFT
31 IMPERVIOUS_SURFACE_SQFT
32 HRA_NAME
33 WATERSHED_NUMBER
34 WATERSHED_NAME
35 PRIORITY_WATERSHED
36 FIREPD
37 WITHIN_TRPA_BNDY
38 LITTORAL
39 AS_LANDVALUE
40 AS_IMPROVALUE
41 AS_SUM
42 TAX_LANDVALUE
43 TAX_IMPROVALUE
44 TAX_SUM
45 TAX_YEAR
46 PAS_ID
47 PAS_NAME
48 INDEX_1987
49 INDEX_1987_HYPERLINK
50 LOCAL_PLAN_HYPERLINK
51 ZONING
52 ZONING_DESCRIPTION
53 SINGLE_FAMILY_DENSITY
54 MULTI_FAMILY_DENSITY
55 TOURIST_ACCOMMODATION_DENSITY
56 BED_BREAKFAST_DENSITY
57 TIME_SHARE_DENSITY
58 COMMERCIAL_FLOOR_AREA_ALLOWED
59 SECONDARY_DWELLING_UNIT_

### Create Parcel Tables

In [8]:
import arcpy
table = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
field_names = [field.name for field in arcpy.ListFields(table)]
# lists the index of the field names in parcel out data
for index, field in enumerate(field_names):
    print (index, field)

0 OBJECTID
1 Shape
2 APN
3 PPNO
4 HSE_NUMBR
5 UNIT_NUMBR
6 STR_DIR
7 STR_NAME
8 STR_SUFFIX
9 APO_ADDRESS
10 PSTL_TOWN
11 PSTL_STATE
12 PSTL_ZIP5
13 OWN_FIRST
14 OWN_LAST
15 OWN_FULL
16 MAIL_ADD1
17 MAIL_ADD2
18 MAIL_CITY
19 MAIL_STATE
20 MAIL_ZIP5
21 JURISDICTION
22 COUNTY
23 OWNERSHIP_TYPE
24 COUNTY_LANDUSE_CODE
25 COUNTY_LANDUSE_DESCRIPTION
26 TRPA_LANDUSE_DESCRIPTION
27 REGIONAL_LANDUSE
28 SOIL_1974
29 SOIL_2003
30 ALLOWABLE_COVERAGE_BAILEY_SQFT
31 IMPERVIOUS_SURFACE_SQFT
32 HRA_NAME
33 WATERSHED_NUMBER
34 WATERSHED_NAME
35 PRIORITY_WATERSHED
36 FIREPD
37 WITHIN_TRPA_BNDY
38 LITTORAL
39 AS_LANDVALUE
40 AS_IMPROVALUE
41 AS_SUM
42 TAX_LANDVALUE
43 TAX_IMPROVALUE
44 TAX_SUM
45 TAX_YEAR
46 PAS_ID
47 PAS_NAME
48 INDEX_1987
49 INDEX_1987_HYPERLINK
50 LOCAL_PLAN_HYPERLINK
51 ZONING
52 ZONING_DESCRIPTION
53 SINGLE_FAMILY_DENSITY
54 MULTI_FAMILY_DENSITY
55 TOURIST_ACCOMMODATION_DENSITY
56 BED_BREAKFAST_DENSITY
57 TIME_SHARE_DENSITY
58 COMMERCIAL_FLOOR_AREA_ALLOWED
59 SECONDARY_DWELLING_UNIT_

In [8]:
import arcpy

def getFieldMappings(fc_in, mapping_list):
    field_mappings = arcpy.FieldMappings()

    for in_field, out_field, out_type in mapping_list:
        field_map = arcpy.FieldMap()
        field_map.addInputField(fc_in, in_field)
        field = field_map.outputField
        field.name = out_field
        field.type = out_type
        field_map.outputField = field
        field_mappings.addFieldMap(field_map)
        print("added {} field map object".format(field.name))
        del field, field_map

    return field_mappings

# input table and output path
table = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb\\Parcel_Master"
outpath = "F:\\GIS\\ParcelUpdate\\" + w_folder + "\\Workspace\\Parcel_Update_Workspace.gdb"

# list of all Parcel fields 
fieldListMaster = (
        "APN", "PPNO", "JURISDICTION", "HSE_NUMBR", "UNIT_NUMBR", "STR_DIR", "STR_NAME", "STR_SUFFIX", "APO_ADDRESS",
        "PSTL_TOWN", "PSTL_STATE", "PSTL_ZIP5", "OWN_FIRST", "OWN_LAST", "OWN_FULL", "MAIL_ADD1", "MAIL_ADD2", "MAIL_CITY",
        "MAIL_STATE", "MAIL_ZIP5", "AS_LANDVALUE", "AS_IMPROVALUE", "AS_SUM", "TAX_LANDVALUE", "TAX_IMPROVALUE", "TAX_SUM",
        "TAX_YEAR", "OWNERSHIP_TYPE", "TRPA_LANDUSE_DESCRIPTION", "COUNTY_LANDUSE_CODE", "COUNTY_LANDUSE_DESCRIPTION", "SOIL_1974",
        "SOIL_2003", "HRA_NAME", "WATERSHED_NUMBER", "WATERSHED_NAME",  "PRIORITY_WATERSHED", "PAS_ID", "PAS_NAME", "FIREPD",
        "BMP_STATUS", "WITHIN_TRPA_BNDY", "LITTORAL", "PARCEL_ACRES", "PARCEL_SQFT", "INDEX_1987",
        "INDEX_1987_HYPERLINK", "LOCAL_PLAN_HYPERLINK", "IMPERVIOUS_SURFACE_SQFT", "ALLOWABLE_COVERAGE_BAILEY_SQFT",
        "ZONING", "ZONING_DESCRIPTION", "SUBDISTRICT", "REGIONAL_LANDUSE", "SINGLE_FAMILY_DENSITY", "MULTI_FAMILY_DENSITY",
        "TOURIST_ACCOMMODATION_DENSITY", "BED_BREAKFAST_DENSITY", "TIME_SHARE_DENSITY", "COMMERCIAL_FLOOR_AREA_ALLOWED",
        "SECONDARY_DWELLING_UNIT_ALLOWED", "OVERLAY", "MAX_CP_RES_DENSITY", "MAX_CP_TAU_DENSITY", "SPECIAL_PLAN_AREA_OVERLAY",
        "TOWN_CENTER", "SPECIAL_AREA", "SPECIAL_PLANNING_DISTRICT", "SPECIAL_PLANNING_DISTRICT_TYPE", "LOCATION_TO_TOWNCENTER",
        "COUNTY")

#Export Parcel_Master table
table_out = "Parcel_Master_Table"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

master_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "JURISDICTION", "Text"),
               (fieldListMaster[3], "HSE_NUMBR", "Short Integer"),
               (fieldListMaster[4], "UNIT_NUMBR", "Text"),
               (fieldListMaster[5], "STR_DIR", "Text"),
               (fieldListMaster[6], "STR_NAME", "Text"),
               (fieldListMaster[7], "STR_SUFFIX", "Text"),
               (fieldListMaster[8], "APO_ADDRESS", "Text"),
               (fieldListMaster[9], "PSTL_TOWN", "Text"),
               (fieldListMaster[10], "PSTL_STATE", "Text"),
               (fieldListMaster[11], "PSTL_ZIP5", "Text"),
               (fieldListMaster[12], "OWN_FIRST", "Text"),
               (fieldListMaster[13], "OWN_LAST", "Text"),
               (fieldListMaster[14], "OWN_FULL", "Text"),
               (fieldListMaster[15], "MAIL_ADD1", "Text"),
               (fieldListMaster[16], "MAIL_ADD2", "Text"),
               (fieldListMaster[17], "MAIL_CITY", "Text"),
               (fieldListMaster[18], "MAIL_STATE", "Text"),
               (fieldListMaster[19], "MAIL_ZIP5", "Text"),
               (fieldListMaster[20], "AS_LANDVALUE", "Long"),
               (fieldListMaster[21], "AS_IMPROVALUE", "Long"),
               (fieldListMaster[22], "AS_SUM", "Long"),
               (fieldListMaster[23], "TAX_LANDVALUE", "Long"),
               (fieldListMaster[24], "TAX_IMPROVALUE", "Long"),
               (fieldListMaster[25], "TAX_SUM", "Long"),
               (fieldListMaster[26], "TAX_YEAR", "Text"),
               (fieldListMaster[27], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[28], "TRPA_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[29], "COUNTY_LANDUSE", "Text"),
               (fieldListMaster[30], "COUNTY_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[31], "SOIL_1974", "Text"),
               (fieldListMaster[32], "SOIL_2003", "Text"),
               (fieldListMaster[33], "HRA_NAME", "Text"),
               (fieldListMaster[34], "WATERSHED_NUMBER", "Short"),
               (fieldListMaster[35], "WATERSHED_NAME", "Text"),
               (fieldListMaster[36], "PRIORITY_WATERSHED", "Text"),
               (fieldListMaster[37], "PAS_ID", "Text"),
               (fieldListMaster[38], "PAS_NAME", "Text"),
               (fieldListMaster[39], "FIREPD", "Text"),
               # (fieldListMaster[40], "BMP_STATUS", "Text"),
               (fieldListMaster[41], "WITHIN_TRPA_BNDY", "Text"),
               (fieldListMaster[42], "LITTORAL", "Text"),
               (fieldListMaster[43], "PARCEL_ACRES", "Double"),
               (fieldListMaster[44], "PARCEL_SQFT", "Double"),
               (fieldListMaster[45], "INDEX_1987", "Text"),
               (fieldListMaster[46], "INDEX_1987_HYPERLINK", "Text"),
               (fieldListMaster[47], "LOCAL_PLAN_HYPERLINK", "Text"),
               (fieldListMaster[48], "IMPERVIOUS_SURFACE_SQFT", "Double"),
               (fieldListMaster[49], "ALLOWABLE_COVERAGE_BAILEY_SQFT", "Double"),
               (fieldListMaster[50], "ZONING", "Text"),
               (fieldListMaster[51], "ZONING_DESCRIPTION", "Text"),
               (fieldListMaster[53], "REGIONAL_LANDUSE", "Text"),
               (fieldListMaster[54], "SINGLE_FAMILY_DENSITY", "Text"),
               (fieldListMaster[55], "MULTI_FAMILY_DENSITY", "Text"),
               (fieldListMaster[56], "TOURIST_ACCOMMODATION_DENSITY", "Text"),
               (fieldListMaster[57], "BED_BREAKFAST_DENSITY", "Text"),
               (fieldListMaster[58], "TIME_SHARE_DENSITY", "Text"),
               (fieldListMaster[59], "COMMERCIAL_FLOOR_AREA_ALLOWED", "Text"),
               (fieldListMaster[60], "SECONDARY_DWELLING_UNIT_ALLOWED", "Text"),
               (fieldListMaster[61], "OVERLAY", "Text"),
               (fieldListMaster[62], "MAX_CP_RES_DENSITY", "Text"),
               (fieldListMaster[63], "MAX_CP_TAU_DENSITY", "Text"),
               (fieldListMaster[64], "SPECIAL_PLAN_AREA_OVERLAY", "Text"),
               (fieldListMaster[65], "TOWN_CENTER", "Text"),
               (fieldListMaster[66], "SPECIAL_AREA", "Text"),
               (fieldListMaster[67], "SPECIAL_PLANNING_DISTRICT", "Text"),
               (fieldListMaster[68], "SPECIAL_PLANNING_DISTRICT_TYPE", "Text"),
               (fieldListMaster[69], "LOCATION_TO_TOWNCENTER", "Text"),
               (fieldListMaster[70], "COUNTY", "Text")]

mapped = getFieldMappings(table, master_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


#Export Parcel_Address Table
table_out = "Parcel_Address"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

address_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "JURISDICTION", "Text"),
               (fieldListMaster[3], "HSE_NUMBR", "Short Integer"),
               (fieldListMaster[4], "UNIT_NUMBR", "Text"),
               (fieldListMaster[5], "STR_DIR", "Text"),
               (fieldListMaster[6], "STR_NAME", "Text"),
               (fieldListMaster[7], "STR_SUFFIX", "Text"),
               (fieldListMaster[8], "APO_ADDRESS", "Text"),
               (fieldListMaster[9], "PSTL_TOWN", "Text"),
               (fieldListMaster[10], "PSTL_STATE", "Text"),
               (fieldListMaster[11], "PSTL_ZIP5", "Text")]

mapped = getFieldMappings(table, address_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


#Export Parcel_Owner Table
table_out = "Parcel_Owner"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

owner_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "JURISDICTION", "Text"),
               (fieldListMaster[12], "OWN_FIRST", "Text"),
               (fieldListMaster[13], "OWN_LAST", "Text"),
               (fieldListMaster[14], "OWN_FULL", "Text"),
               (fieldListMaster[15], "MAIL_ADD1", "Text"),
               (fieldListMaster[16], "MAIL_ADD2", "Text"),
               (fieldListMaster[17], "MAIL_CITY", "Text"),
               (fieldListMaster[18], "MAIL_STATE", "Text"),
               (fieldListMaster[19], "MAIL_ZIP5", "Text")]

mapped = getFieldMappings(table, owner_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


# Export Parcel_Value Table
table_out = "Parcel_Value"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

value_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "JURISDICTION", "Text"),
               (fieldListMaster[20], "AS_LANDVALUE", "Long"),
               (fieldListMaster[21], "AS_IMPROVALUE", "Long"),
               (fieldListMaster[22], "AS_SUM", "Long"),
               (fieldListMaster[23], "TAX_LANDVALUE", "Long"),
               (fieldListMaster[24], "TAX_IMPROVALUE", "Long"),
               (fieldListMaster[25], "TAX_SUM", "Long"),
               (fieldListMaster[26], "TAX_YEAR", "Text"),
               (fieldListMaster[9], "PSTL_TOWN", "Text"),
               (fieldListMaster[10], "PSTL_STATE", "Text"),
               (fieldListMaster[11], "PSTL_ZIP5", "Text"),
               (fieldListMaster[17], "MAIL_CITY", "Text"),
               (fieldListMaster[18], "MAIL_STATE", "Text"),
               (fieldListMaster[19], "MAIL_ZIP5", "Text"),
               (fieldListMaster[27], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[43], "PARCEL_ACRES", "Double"),
               (fieldListMaster[44], "PARCEL_SQFT", "Double")]

mapped = getFieldMappings(table, value_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


# Export Parcel_APO Table
table_out = "Parcel_APO"

print ("Creating Field Map Objects for %s table" %(table_out))

if arcpy.Exists(outpath + "/" + table_out):
    arcpy.Delete_management(outpath + "/" + table_out)

apo_dict = [(fieldListMaster[0], "APN", "Text"),
               (fieldListMaster[1], "PPNO", "Double"),
               (fieldListMaster[2], "JURISDICTION", "Text"),
               (fieldListMaster[14], "OWN_FULL", "Text"),
               (fieldListMaster[8], "APO_ADDRESS", "Text"),
               (fieldListMaster[27], "OWNERSHIP_TYPE", "Text"),
               (fieldListMaster[28], "TRPA_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[29], "COUNTY_LANDUSE", "Text"),
               (fieldListMaster[30], "COUNTY_LANDUSE_DESCRIPTION", "Text"),
               (fieldListMaster[31], "SOIL_1974", "Text"),
               (fieldListMaster[32], "SOIL_2003", "Text"),
               (fieldListMaster[33], "HRA_NAME", "Text"),
               (fieldListMaster[34], "WATERSHED_NUMBER", "Short"),
               (fieldListMaster[35], "WATERSHED_NAME", "Text"),
               (fieldListMaster[36], "PRIORITY_WATERSHED", "Text"),
               (fieldListMaster[37], "PAS_ID", "Text"),
               (fieldListMaster[38], "PAS_NAME", "Text"),
               (fieldListMaster[39], "FIREPD", "Text"),
               (fieldListMaster[41], "WITHIN_TRPA_BNDY", "Text"),
               (fieldListMaster[42], "LITTORAL", "Text"),
               (fieldListMaster[43], "PARCEL_ACRES", "Double"),
               (fieldListMaster[44], "PARCEL_SQFT", "Double"),]

mapped = getFieldMappings(table, apo_dict)

arcpy.TableToTable_conversion(table, outpath, table_out, "", mapped)

print ("Exported %s table" %(table_out))


Creating Field Map Objects for Parcel_Master_Table table
added APN field map object
added PPNO field map object
added JURISDICTION field map object
added HSE_NUMBR field map object
added UNIT_NUMBR field map object
added STR_DIR field map object
added STR_NAME field map object
added STR_SUFFIX field map object
added APO_ADDRESS field map object
added PSTL_TOWN field map object
added PSTL_STATE field map object
added PSTL_ZIP5 field map object
added OWN_FIRST field map object
added OWN_LAST field map object
added OWN_FULL field map object
added MAIL_ADD1 field map object
added MAIL_ADD2 field map object
added MAIL_CITY field map object
added MAIL_STATE field map object
added MAIL_ZIP5 field map object
added AS_LANDVALUE field map object
added AS_IMPROVALUE field map object
added AS_SUM field map object
added TAX_LANDVALUE field map object
added TAX_IMPROVALUE field map object
added TAX_SUM field map object
added TAX_YEAR field map object
added OWNERSHIP_TYPE field map object
added TRPA_

## QA/QC

### Check for NULL

### Check Topology

### Check for Duplicates

### Check that APN == PPNO 

## Issues Log

###  Spring 2019
* Check for Blank Land Use Values in El Dorado

* Merge Duplicates in El Dorado and Placer data

* Check for '0' in Placer PPNO

* Add # to valid Unit number and Null to ''

* Null to '' for all fields

* Script Select by Location of CSLT Values in Jurisdiction field, County field == EL, CC, PL, DG 

* Generate List of New APNs for Adelle

* Add Land Use Calculation NOT OWNERSHIP_TYPE = 'PRIVATE' AND TRPA_LANDUSE_DESCRIPTION = 'Vacant' 

* Fix Duplicates at the beginning of the script

* If Mail1 == Mail2 Then Mail2 is NULL

## References

### Accessing data using cursors
* http://pro.arcgis.com/en/pro-app/arcpy/get-started/data-access-using-cursors.htm

### Replacement for the field calculator
* https://gis.stackexchange.com/questions/177923/using-updatecursor-for-joined-field-in-arcpy/178168#178168
* https://community.esri.com/blogs/richard_fairhurst/2014/11/08/turbo-charging-data-manipulation-with-python-cursors-and-dictionaries

### Which way is faster?
* https://gis.stackexchange.com/questions/195197/which-is-the-faster-way-to-copy-data-to-another-feature-class-feature-class-to?utm_medium=organic&utm_source=google_rich_qa&utm_campaign=google_rich_qa

### Full Address Calculation
* https://community.esri.com/thread/42173
* https://bit.ly/2HsQL9x

### How to create the attribute join and update function:
* https://gis.stackexchange.com/questions/177923/using-updatecursor-for-joined-field-in-arcpy/178168?utm_medium=organic&utm_source=google_rich_qa&utm_campaign=google_rich_qa

### How to create the centroid feature layer:
* http://pro.arcgis.com/en/pro-app/arcpy/data-access/featureclasstonumpyarray.htm
* http://pro.arcgis.com/en/pro-app/arcpy/data-access/numpyarraytofeatureclass.htm

### How to use the Spatial Join tool:
* http://pro.arcgis.com/en/pro-app/tool-reference/analysis/spatial-join.htm

### How to use the Select by Location tool:
* http://desktop.arcgis.com/en/arcmap/10.3/tools/data-management-toolbox/select-layer-by-location.htm

### How to use the Calculate field tool:
* http://pro.arcgis.com/en/pro-app/tool-reference/data-management/calculate-field.htm

### How to Simplify Polygons
* http://pro.arcgis.com/en/pro-app/tool-reference/cartography/simplify-polygon.htm